# 02 — Exploit Likelihood Prediction

**Status: research notebook (EXPERIMENTAL).** Nothing here is wired into OffensiveX. Output is **never** a "probability of compromise" — see §18 and `docs/roadmap/ml-integration-contract.md`.

## Purpose
Test whether public vulnerability metadata known shortly after a CVE is published can **rank** CVEs by how likely they are to be *added to CISA's Known Exploited Vulnerabilities (KEV) catalog within a fixed horizon* — evaluated on **later** CVEs than the training data.

## Concepts that must not be conflated (and what this notebook uses)
| Concept | What it is | Used here as |
|---|---|---|
| **CVSS severity** | Technical impact/ease *if* exploited (deterministic from metrics) | a *feature* |
| **Published PoC / exploit code** | Someone wrote exploit code | only a weak text hint (`kw_exploit_public` flag) — **no PoC dataset was verified, so no PoC label exists** |
| **KEV listing** | CISA has evidence of exploitation in the wild *and* a clear remediation (US-federal-centric curation) | **the target label**, with a time horizon |
| **Observed exploitation in the wild** | Real attacks happened | *unobserved in general*; KEV is an incomplete proxy |
| **EPSS** | FIRST's model estimate of 30-day exploitation probability (it already uses KEV-like signals) | **never a feature or label**; shown only as a *hindsight-contaminated reference comparator* |

## Target (documented definition)
`y = 1` iff the CVE was added to KEV **on or before `published + LABEL_HORIZON_DAYS`** (default 90 days; KEV additions *before* NVD publication count as positive). `y = 0` means **"not recorded in KEV within the window"** — *not* "never exploited": unreported exploitation, late KEV additions (after the window) and non-US-relevant exploitation all land in the negative class. CVEs published less than one horizon before the data snapshot are **dropped** (right-censoring: their window is not fully observed).

## Research questions
1. **RQ1.** Can metadata known at/near publication rank KEV-bound CVEs better than chance and better than CVSS score alone, on a chronological holdout?
2. **RQ2.** Are predicted probabilities calibratable (calibration fitted on validation only)?
3. **RQ3.** Which operating thresholds / review budgets (top-k) give acceptable recall, and at what precision?
4. **RQ4.** How much would leaked fields (reference counts, current EPSS) inflate results?
5. **RQ5.** Where does the model fail (false positives/negatives; subgroups; over time)?

If no valid target can be established (too few positives), the notebook **stops before training** and writes `blocked_report.json` (see §6).

## 2. Environment and dependency setup
Nothing is installed silently. Locally: `pip install numpy pandas scikit-learn matplotlib seaborn joblib pyarrow`. `shap` is not used.

In [ ]:
import os, sys, re, io, gc, json, gzip, math, time, hashlib, platform, warnings, random, importlib.util
import urllib.request, urllib.error
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

try:
    from IPython.display import display, Markdown
except ImportError:  # plain-python fallback so the code also runs as a script
    display, Markdown = print, str

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

# ---- environment detection (used ONLY for paths/setup, never to change the methodology) ----
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
ENVIRONMENT = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

# ---- dependency check (nothing is installed silently) ----
_REQUIRED = ["numpy", "pandas", "sklearn", "matplotlib", "seaborn", "joblib"]
_OPTIONAL = {"pyarrow": "needed only to read .parquet inputs", "shap": "optional; permutation importance is used if absent"}
_missing = [m for m in _REQUIRED if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(
        f"Missing required packages: {_missing}. Install with: pip install numpy pandas scikit-learn matplotlib seaborn joblib pyarrow"
    )
OPTIONAL_AVAILABLE = {m: importlib.util.find_spec(m) is not None for m in _OPTIONAL}


def _dependency_versions():
    out = {"python": sys.version.split()[0], "platform": platform.platform()}
    for name in ["numpy", "pandas", "sklearn", "scipy", "matplotlib", "seaborn", "joblib", "pyarrow", "shap"]:
        try:
            out[name] = importlib.import_module(name).__version__
        except Exception:
            out[name] = None
    return out


DEPENDENCY_VERSIONS = _dependency_versions()


def resolve_default_dirs(cfg):
    """Fill OUTPUT_ROOT / CACHE_DIR when left as None. Override anytime via CONFIG or the OFFENSIVEX_ML_OUTPUT env var."""
    if cfg.get("OUTPUT_ROOT") is None:
        if os.environ.get("OFFENSIVEX_ML_OUTPUT"):
            cfg["OUTPUT_ROOT"] = os.environ["OFFENSIVEX_ML_OUTPUT"]
        elif IN_KAGGLE:
            cfg["OUTPUT_ROOT"] = "/kaggle/working/artifacts/ml-research"
        elif IN_COLAB:
            cfg["OUTPUT_ROOT"] = "/content/artifacts/ml-research"
        else:  # local Jupyter: <repo>/artifacts/ml-research (git-ignored), whether launched from the repo root or notebooks/
            base = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
            cfg["OUTPUT_ROOT"] = str(base / "artifacts" / "ml-research")
    if cfg.get("CACHE_DIR") is None:
        cfg["CACHE_DIR"] = str(Path(cfg["OUTPUT_ROOT"]).parent / "data_cache")
    return cfg


def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

## 3. Dataset sources, access instructions, licensing and limitations

| Source | Verified identifier / URL (2026-10-14) | Access | License / terms | Role |
|---|---|---|---|---|
| **Kaggle "NVD CVE CVSS KEV EPSS Vulnerabilities 2020-2026"** | `afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026` → `nvd_cve_records.parquet` (201,542 CVEs; columns `kev_is_known_exploited`, `kev_dateadded`, `epss_score`, `epss_percentile`, `cvss_v31_*`, …; built 2026-05-02; KEV rows 1,585, 1,039 matching CVEs in range) | Attach via Kaggle **Input** panel; internet may stay **off** | CC0 (Kaggle listing); underlying NVD / CISA KEV / FIRST EPSS | Primary (offline) |
| **CISA KEV catalog** | `https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv` (columns incl. `cveID`, `dateAdded`; 1,741 rows on 2026-10-14) | public HTTPS | CISA license linked on <https://www.cisa.gov/known-exploited-vulnerabilities-catalog> | label source (official-feed route) |
| **FIRST EPSS** | `https://epss.cyentia.com/epss_scores-current.csv.gz` (`cve,epss,percentile`; header carries `model_version`, `score_date`; ≈389k rows) | public HTTPS | free to use with attribution to FIRST (<https://www.first.org/epss/>) | reference comparator only |
| **NVD CVE JSON 2.0 feeds** | `https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-<year>.json.gz` | public HTTPS | public NVD data | features (official-feed route) |

**Targets considered and rejected:** (a) *EPSS as label* — it is a model's output, and it would turn this into "approximating EPSS", not exploitation prediction; (b) *Exploit-DB/PoC existence* — no verified, license-checked dataset was attached; (c) *"in KEV ever"* — unbounded window, confounded by CVE age (older CVEs have had longer to be added).

**Label limitations / biases:** KEV is incomplete and US-federal-centric, lists only CVEs with a CVE ID and remediation guidance, and its addition date reflects CISA's process timing, not the first attack. Positives are rare (<1%) → precision is intrinsically low. Features come from the *current* NVD record (CVSS may have been assigned days–weeks after publication). EPSS "current" is hindsight-contaminated (its inputs include post-publication exploitation evidence). Association ≠ causation.

**Colab/local:** download the parquet via the Kaggle API (token in Colab *Secrets*; never in a cell) and set `DATASET_PATH`, or set `ALLOW_NETWORK=True` to fetch NVD feeds + KEV + EPSS into `CACHE_DIR` (cached; delete to refresh).

## 4. Configuration variables
**Change if needed:** `DATASET_PATH`, `ALLOW_NETWORK`, `NVD_YEARS`, `OUTPUT_ROOT`. **Methodology knobs:** `LABEL_HORIZON_DAYS`, `TIME_SPLIT_QUANTILES`, `RECALL_TARGET`. `FAST_DEV=True` keeps all positives but subsamples negatives (changes prevalence — debugging only; artifacts flagged non-reportable).

In [ ]:
CONFIG = dict(
    DATASET_PATH=None, ALLOW_NETWORK=False, NVD_YEARS=list(range(2020, 2027)), OUTPUT_ROOT=None, CACHE_DIR=None,
    EXPERIMENT_DIR="exploit_likelihood", FAST_DEV=False, FAST_DEV_NEGATIVES=40_000,
    SEED=42,
    LABEL_HORIZON_DAYS=90,              # positive = added to KEV within this many days after NVD publication
    SNAPSHOT_DATE=None,                 # None = max(published, KEV dateAdded) in the data; or e.g. "2026-05-02"
    TIME_SPLIT_QUANTILES=(0.60, 0.80),  # chronological cut points over eligible CVEs; an embargo of LABEL_HORIZON_DAYS precedes each cut
    MIN_POSITIVES_TOTAL=150, MIN_POSITIVES_PER_SPLIT=30,
    THRESHOLD_POLICY="recall_target",   # "recall_target" (highest precision with val recall >= RECALL_TARGET) or "max_f1"
    RECALL_TARGET=0.80,
    TOPK_FRACTIONS=(0.01, 0.02, 0.05, 0.10),
    N_BOOTSTRAP=500, TFIDF_MAX_FEATURES=20_000, PERM_IMPORTANCE_ROWS=15_000, PLOT_SAMPLE_ROWS=20_000, N_JOBS=-1,
)
resolve_default_dirs(CONFIG)
set_seeds(CONFIG["SEED"])

In [ ]:
# ======================= CVSS helpers: vector parsing, official calculators, severity bands =======================
# Formulas: FIRST CVSS v3.1 specification section 7 (https://www.first.org/cvss/v3-1/specification-document),
# v3.0 (https://www.first.org/cvss/v3.0/specification-document), v2 (https://www.first.org/cvss/v2/guide).
# CVSS v4.0 scoring uses a macro-vector lookup table and is deliberately NOT re-implemented here.
CVSS_VERSIONS = {"v40": "4.0", "v31": "3.1", "v30": "3.0", "v2": "2.0"}


def parse_cvss_vector(vec):
    """'CVSS:3.1/AV:N/...' or (v2) 'AV:N/AC:L/...' -> (version, {metric: value}); None if unparsable."""
    if not isinstance(vec, str) or not vec:
        return None
    version, metrics = None, {}
    for part in vec.split("/"):
        if ":" not in part:
            return None
        k, v = part.split(":", 1)
        if k == "CVSS":
            version = v
        else:
            metrics[k] = v
    return (version or "2.0"), metrics


_W31 = {"AV": {"N": 0.85, "A": 0.62, "L": 0.55, "P": 0.2}, "AC": {"L": 0.77, "H": 0.44},
        "UI": {"N": 0.85, "R": 0.62}, "CIA": {"H": 0.56, "L": 0.22, "N": 0.0},
        "PR_U": {"N": 0.85, "L": 0.62, "H": 0.27}, "PR_C": {"N": 0.85, "L": 0.68, "H": 0.5}}


def _roundup_31(x):
    i = int(round(x * 100000))
    return i / 100000.0 if i % 10000 == 0 else (math.floor(i / 10000) + 1) / 10.0


def cvss3_base_score(vector):
    """Official CVSS v3.0/v3.1 base score from a vector string (None if the vector is invalid)."""
    p = parse_cvss_vector(vector)
    if p is None or p[0] not in ("3.0", "3.1"):
        return None
    ver, m = p
    try:
        changed = m["S"] == "C"
        iss = 1 - (1 - _W31["CIA"][m["C"]]) * (1 - _W31["CIA"][m["I"]]) * (1 - _W31["CIA"][m["A"]])
        impact = 7.52 * (iss - 0.029) - 3.25 * (iss - 0.02) ** 15 if changed else 6.42 * iss
        pr = (_W31["PR_C"] if changed else _W31["PR_U"])[m["PR"]]
        expl = 8.22 * _W31["AV"][m["AV"]] * _W31["AC"][m["AC"]] * pr * _W31["UI"][m["UI"]]
    except KeyError:
        return None
    if impact <= 0:
        return 0.0
    raw = min((impact + expl) * (1.08 if changed else 1.0), 10)
    return _roundup_31(raw) if ver == "3.1" else math.ceil(raw * 10 - 1e-9) / 10.0


def cvss2_base_score(vector):
    p = parse_cvss_vector(vector)
    if p is None or p[0] != "2.0":
        return None
    m = p[1]
    try:
        av, ac, au = {"L": 0.395, "A": 0.646, "N": 1.0}[m["AV"]], {"H": 0.35, "M": 0.61, "L": 0.71}[m["AC"]], {"M": 0.45, "S": 0.56, "N": 0.704}[m["Au"]]
        cia = {"N": 0.0, "P": 0.275, "C": 0.660}
        impact = 10.41 * (1 - (1 - cia[m["C"]]) * (1 - cia[m["I"]]) * (1 - cia[m["A"]]))
    except KeyError:
        return None
    expl = 20 * av * ac * au
    f = 0 if impact == 0 else 1.176
    return round(((0.6 * impact) + (0.4 * expl) - 1.5) * f + 1e-9, 1)


OFFICIAL_CALCULATORS = {"v31": cvss3_base_score, "v30": cvss3_base_score, "v2": cvss2_base_score}  # v40: none implemented


def cvss_severity(score, version):
    """Qualitative rating per the CVSS specification for the given version. Returns an upper-case label or None."""
    if score is None or (isinstance(score, float) and math.isnan(score)):
        return None
    if version == "v2":  # v2 has no NONE/CRITICAL band
        return "LOW" if score < 4.0 else "MEDIUM" if score < 7.0 else "HIGH"
    if score == 0.0:
        return "NONE"
    return "LOW" if score < 4.0 else "MEDIUM" if score < 7.0 else "HIGH" if score < 9.0 else "CRITICAL"


# Mapping to OffensiveX's finding severity enum (backend/src/offensivex/db/models.py::FindingSeverity).
CVSS_TO_OFFENSIVEX_SEVERITY = {"NONE": "info", "LOW": "low", "MEDIUM": "medium", "HIGH": "high", "CRITICAL": "critical"}
CVE_ID_RE = re.compile(r"^CVE-\d{4}-\d{4,}$")

In [ ]:
# ======================= dataset discovery, loading, fingerprinting =======================
# Supported inputs (auto-detected, in this order):
#   1. A "canonical table" named nvd_cve_records.parquet / .csv  (Kaggle: afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026)
#   2. Official NVD CVE JSON 2.0 yearly feeds named nvdcve-2.0-<year>.json.gz found locally
#   3. Official NVD feeds downloaded into CACHE_DIR -- ONLY when CONFIG["ALLOW_NETWORK"] is True
# Nothing is downloaded when a local source is found, and nothing is copied out of /kaggle/input.

SUPPORTED_SUFFIXES = (".parquet", ".csv", ".csv.gz", ".json", ".json.gz", ".jsonl", ".jsonl.gz")
NVD_FEED_URL = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{year}.json.gz"
KEV_URL = "https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv"
EPSS_URL = "https://epss.cyentia.com/epss_scores-current.csv.gz"
KEV_NAMES = ("known_exploited_vulnerabilities.csv", "cisa_kev_catalog.csv")
EPSS_NAMES = ("epss_scores-current.csv.gz", "first_epss_current.csv")
_SKIP_DIRS = {"node_modules", "__pycache__", "artifacts", ".venv", "venv", "site-packages", "sample_data"}


class DataNotFoundError(RuntimeError):
    pass


def _roots_to_scan():
    if CONFIG["DATASET_PATH"]:
        return [Path(CONFIG["DATASET_PATH"]).expanduser()]
    roots = []
    if IN_KAGGLE:
        roots.append(Path("/kaggle/input"))
    roots += [Path(CONFIG["CACHE_DIR"]), Path.cwd(), Path.cwd() / "data"]
    if IN_COLAB:
        roots.append(Path("/content"))
    seen, out = set(), []
    for r in roots:
        if r.exists() and str(r.resolve()) not in seen:
            seen.add(str(r.resolve()))
            out.append(r)
    return out


def list_input_files(roots, max_depth=4, limit=500):
    found = []
    for root in roots:
        if root.is_file():
            found.append(root)
            continue
        base = len(root.resolve().parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).resolve().parts) - base >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in _SKIP_DIRS]
            for fn in filenames:
                if fn.lower().endswith(SUPPORTED_SUFFIXES):
                    found.append(Path(dirpath) / fn)
            if len(found) >= limit:
                return found
    return found


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def http_download(url, dest, tries=4, timeout=120, backoff=5):
    """Cached, retrying, rate-limit-aware download (honours Retry-After on 429/503). Atomic write."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  cache hit: {dest.name} ({dest.stat().st_size/1e6:.1f} MB) -- delete it to refresh")
        return dest
    part = Path(str(dest) + ".part")
    last, wait = None, backoff
    for attempt in range(1, tries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "OffensiveX-ml-research/1.0 (research notebook)"})
            with urllib.request.urlopen(req, timeout=timeout) as r, open(part, "wb") as f:
                while block := r.read(1 << 20):
                    f.write(block)
            part.replace(dest)
            print(f"  downloaded {url} -> {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
            return dest
        except urllib.error.HTTPError as e:
            last = e
            if e.code in (400, 401, 403, 404):
                raise RuntimeError(f"HTTP {e.code} for {url} (not retryable)") from e
            ra = e.headers.get("Retry-After") if e.headers else None
            wait = int(ra) if ra and ra.isdigit() else backoff * attempt * (2 if e.code in (429, 503) else 1)
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            last, wait = e, backoff * attempt
        print(f"  attempt {attempt}/{tries} failed for {url}: {last}; retrying in {wait}s")
        time.sleep(wait)
    raise RuntimeError(f"Could not download {url} after {tries} attempts: {last}")


# ----------------------------- NVD CVE JSON 2.0 parsing -----------------------------
_CVSS_KEYS = {"v40": "cvssMetricV40", "v31": "cvssMetricV31", "v30": "cvssMetricV30", "v2": "cvssMetricV2"}
_CVSS_FIELDS = ["vector", "base_score", "base_severity", "source", "type", "exploitability_score", "impact_score"]
BASE_COLUMNS = ["cve_id", "source_identifier", "published_date", "last_modified_date", "vuln_status", "description_en",
                "reference_count", "weakness_count", "cpe_match_count", "primary_cwe_id"]
CANON_COLUMNS = BASE_COLUMNS + [f"cvss_{v}_{f}" for v in _CVSS_KEYS for f in _CVSS_FIELDS]


def _count_cpe_matches(configs):
    n = 0
    for conf in configs or []:
        stack = list(conf.get("nodes", []))
        while stack:
            node = stack.pop()
            n += len(node.get("cpeMatch", []))
            stack.extend(node.get("children", []))
    return n


def parse_nvd_record(rec):
    c = rec["cve"]
    row = {"cve_id": c.get("id"), "source_identifier": c.get("sourceIdentifier"),
           "published_date": c.get("published"), "last_modified_date": c.get("lastModified"),
           "vuln_status": c.get("vulnStatus")}
    row["description_en"] = next((d["value"] for d in c.get("descriptions", []) if d.get("lang") == "en"), None)
    row["reference_count"] = len(c.get("references", []))
    cwes = [d["value"] for w in c.get("weaknesses", []) for d in w.get("description", [])]
    row["weakness_count"] = len(set(cwes))
    row["primary_cwe_id"] = cwes[0] if cwes else None
    row["cpe_match_count"] = _count_cpe_matches(c.get("configurations"))
    metrics = c.get("metrics", {})
    for ver, key in _CVSS_KEYS.items():
        entries = metrics.get(key) or []
        m = next((e for e in entries if e.get("type") == "Primary"), entries[0] if entries else None)
        p = f"cvss_{ver}_"
        if m is None:
            continue
        d = m.get("cvssData", {})
        row[p + "vector"] = d.get("vectorString")
        row[p + "base_score"] = d.get("baseScore")
        row[p + "base_severity"] = d.get("baseSeverity") or m.get("baseSeverity")
        row[p + "source"] = m.get("source")
        row[p + "type"] = m.get("type")
        row[p + "exploitability_score"] = m.get("exploitabilityScore")
        row[p + "impact_score"] = m.get("impactScore")
    return row


def parse_nvd_feed(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("format") != "NVD_CVE" or "vulnerabilities" not in data:
        raise ValueError(f"{path} is not an NVD CVE 2.0 feed (format={data.get('format')!r})")
    return pd.DataFrame([parse_nvd_record(r) for r in data["vulnerabilities"]])


def read_kev(path):
    kev = pd.read_csv(path)
    low = {c.lower(): c for c in kev.columns}
    id_col = low.get("cveid") or low.get("cve_id")
    add_col = low.get("dateadded") or low.get("date_added")
    if id_col is None or add_col is None:
        raise ValueError(f"KEV file {path} lacks cveID/dateAdded columns: {list(kev.columns)}")
    out = kev[[id_col, add_col]].rename(columns={id_col: "cve_id", add_col: "kev_dateadded"})
    out["kev_dateadded"] = pd.to_datetime(out["kev_dateadded"], errors="coerce")
    return out.drop_duplicates("cve_id")


def read_epss(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        header = f.readline()
    skip = 1 if header.startswith("#") else 0
    e = pd.read_csv(path, skiprows=skip)
    e.columns = [c.lower() for c in e.columns]
    if "cve" in e.columns:
        e = e.rename(columns={"cve": "cve_id"})
    meta = dict(re.findall(r"(\w+):([^,\s]+)", header)) if skip else {}
    e["epss_score_date"] = meta.get("score_date")
    e["epss_model_version"] = meta.get("model_version")
    return e.rename(columns={"epss": "epss_score", "percentile": "epss_percentile"})


def _find_named(files, names):
    lowered = {f.name.lower(): f for f in files}
    for n in names:
        if n in lowered:
            return lowered[n]
    return None


def _standardise(df):
    df = df.copy()
    for c in ("published_date", "last_modified_date"):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_localize(None).astype("datetime64[ns]")
    if "kev_dateadded" in df.columns:
        df["kev_dateadded"] = pd.to_datetime(df["kev_dateadded"], errors="coerce")
    return df


def _read_table(path):
    p = str(path).lower()
    if p.endswith(".parquet"):
        return pd.read_parquet(path)
    if p.endswith((".csv", ".csv.gz")):
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported file type for a canonical table: {path}")


def load_dataset(need_exploit_signals=False):
    """Returns (DataFrame in canonical schema, metadata dict). Raises DataNotFoundError with instructions."""
    roots = _roots_to_scan()
    print(f"Scanning for inputs under: {[str(r) for r in roots]}")
    files = list_input_files(roots)
    print(f"Found {len(files)} candidate data file(s):")
    for f in files[:25]:
        print(f"  {f}  ({f.stat().st_size/1e6:.1f} MB)")
    if len(files) > 25:
        print(f"  ... and {len(files) - 25} more")

    kind, sources = None, []
    canon = _find_named(files, ("nvd_cve_records.parquet", "nvd_cve_records.csv"))
    explicit = Path(CONFIG["DATASET_PATH"]).expanduser() if CONFIG["DATASET_PATH"] else None
    if canon is not None:
        kind, sources = "canonical_table", [canon]
    elif explicit is not None and explicit.is_file():
        kind, sources = "canonical_table", [explicit]
    else:
        feeds = sorted(f for f in files if (m := re.fullmatch(r"nvdcve-2\.0-(\d{4})\.json\.gz", f.name))
                       and int(m.group(1)) in set(CONFIG["NVD_YEARS"]))
        if feeds:
            kind, sources = "nvd_feeds", feeds
    if kind is None:
        if not CONFIG["ALLOW_NETWORK"]:
            raise DataNotFoundError(
                "No supported dataset found and CONFIG['ALLOW_NETWORK'] is False, so nothing will be downloaded.\n"
                "Do ONE of the following:\n"
                "  Kaggle : add the dataset 'afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026' via the notebook Input panel.\n"
                "  Any    : set CONFIG['DATASET_PATH'] to a folder/file containing nvd_cve_records.parquet|csv or nvdcve-2.0-<year>.json.gz.\n"
                "  Any    : set CONFIG['ALLOW_NETWORK']=True to download the official NVD yearly feeds (+KEV/EPSS) into CACHE_DIR."
            )
        print("No local dataset found; ALLOW_NETWORK=True -> downloading official NVD feeds into the cache directory.")
        sources = [http_download(NVD_FEED_URL.format(year=y), Path(CONFIG["CACHE_DIR"]) / f"nvdcve-2.0-{y}.json.gz")
                   for y in CONFIG["NVD_YEARS"]]
        kind = "nvd_feeds"
        files = files + list(sources)

    extra = []
    if kind == "nvd_feeds":
        parts = []
        for s in sources:
            t0 = time.time()
            parts.append(parse_nvd_feed(s))
            print(f"  parsed {Path(s).name}: {len(parts[-1]):,} CVEs in {time.time()-t0:.0f}s")
        df = pd.concat(parts, ignore_index=True)
        for col in CANON_COLUMNS:
            if col not in df.columns:
                df[col] = np.nan
        df = df.drop_duplicates("cve_id", keep="last")
        if need_exploit_signals:
            kev_f, epss_f = _find_named(files, KEV_NAMES), _find_named(files, EPSS_NAMES)
            if kev_f is None or epss_f is None:
                if not CONFIG["ALLOW_NETWORK"]:
                    raise DataNotFoundError(
                        "Exploit-signal files not found locally (need CISA KEV csv and FIRST EPSS csv.gz) and ALLOW_NETWORK is False.\n"
                        f"  KEV : {KEV_URL}\n  EPSS: {EPSS_URL}\nPlace them in CACHE_DIR or enable ALLOW_NETWORK."
                    )
                kev_f = kev_f or http_download(KEV_URL, Path(CONFIG["CACHE_DIR"]) / "known_exploited_vulnerabilities.csv")
                epss_f = epss_f or http_download(EPSS_URL, Path(CONFIG["CACHE_DIR"]) / "epss_scores-current.csv.gz")
            kev, epss = read_kev(kev_f), read_epss(epss_f)
            df = df.merge(kev, on="cve_id", how="left").merge(epss, on="cve_id", how="left")
            df["kev_is_known_exploited"] = df["kev_dateadded"].notna()
            extra = [kev_f, epss_f]
    else:
        df = _read_table(sources[0])
        if need_exploit_signals:
            for need in ("kev_is_known_exploited", "kev_dateadded", "epss_score"):
                if need not in df.columns:
                    raise DataNotFoundError(f"Canonical table lacks '{need}'; this notebook needs KEV/EPSS columns "
                                            "(use the afr1ste Kaggle dataset or the official-feed route with ALLOW_NETWORK).")
    df = _standardise(df)

    file_info = [{"name": Path(s).name, "bytes": Path(s).stat().st_size, "sha256": sha256_file(s)} for s in list(sources) + extra]
    fingerprint = hashlib.sha256(json.dumps([f["sha256"] for f in file_info] + [len(df)]).encode()).hexdigest()
    source_note = {
        "canonical_table": "Kaggle dataset afr1ste/nvd-cve-cvss-kev-vulnerabilities-2020-2026 (CC0 per Kaggle listing; derived from NVD/CISA KEV/FIRST EPSS)"
                           " or a user-supplied table with the same schema",
        "nvd_feeds": "Official NVD CVE JSON 2.0 yearly feeds (https://nvd.nist.gov/vuln/data-feeds) [+ CISA KEV + FIRST EPSS]",
    }[kind]
    meta = {"source_kind": kind, "source_description": source_note, "files": file_info, "dataset_fingerprint_sha256": fingerprint,
            "n_rows_loaded": int(len(df)), "n_columns": int(df.shape[1]), "environment": ENVIRONMENT,
            "loaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    return df, meta

In [ ]:
# ======================= output directories, plotting and artifact helpers =======================
OUTPUT_DIR = Path(CONFIG["OUTPUT_ROOT"]) / CONFIG["EXPERIMENT_DIR"]
PLOT_DIR = OUTPUT_DIR / "plots"
for _d in (OUTPUT_DIR, PLOT_DIR, Path(CONFIG["CACHE_DIR"])):
    _d.mkdir(parents=True, exist_ok=True)
SAVED_PLOTS = []
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 130, "axes.titlesize": 12, "axes.labelsize": 10})


def savefig(name, fig=None):
    """Save the figure to PLOT_DIR/<name>.png AND display it. Every plot in this notebook goes through here."""
    fig = fig or plt.gcf()
    fig.tight_layout()
    path = PLOT_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    SAVED_PLOTS.append(path.name)
    plt.show()
    plt.close(fig)
    return path


def write_json(path, obj):
    def _default(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return None if np.isnan(o) else float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (pd.Timestamp, datetime)):
            return o.isoformat()
        return str(o)
    Path(path).write_text(json.dumps(obj, indent=2, default=_default), encoding="utf-8")
    return path


_SECRET_RE = re.compile(r"(?i)(api[_-]?key|secret|passw(?:or)?d|token|authorization)\"?\s*[:=]\s*\"?[A-Za-z0-9_\-/+=]{12,}")


def scan_for_secrets(directory):
    """Fail loudly if any exported text artifact looks like it contains a credential."""
    hits = []
    for p in Path(directory).rglob("*"):
        if p.is_file() and p.suffix.lower() in {".json", ".md", ".csv", ".txt"}:
            for m in _SECRET_RE.finditer(p.read_text(encoding="utf-8", errors="ignore")):
                hits.append((p.name, m.group(0)[:40]))
    if hits:
        raise RuntimeError(f"Possible secrets found in exported artifacts: {hits}")
    return True


def export_common(tables=None, extra_config=None):
    """Writes the metadata every experiment must ship: dataset metadata+fingerprint, dependency versions, config, exclusion ledger, tables."""
    write_json(OUTPUT_DIR / "dataset_metadata.json", DATA_META)
    write_json(OUTPUT_DIR / "dependency_versions.json", DEPENDENCY_VERSIONS)
    cfg = {k: v for k, v in CONFIG.items() if k not in ("OUTPUT_ROOT", "CACHE_DIR")}
    cfg["DATASET_PATH"] = Path(cfg["DATASET_PATH"]).name if cfg.get("DATASET_PATH") else None  # no machine-specific paths in artifacts
    write_json(OUTPUT_DIR / "experiment_config.json", {**cfg, **(extra_config or {}), "environment": ENVIRONMENT})
    pd.DataFrame(EXCLUSION_LEDGER).to_csv(OUTPUT_DIR / "exclusion_ledger.csv", index=False)
    for name, t in (tables or {}).items():
        t.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)


EXCLUSION_LEDGER = []


def ledger(step, reason, before, after):
    """Record every row exclusion -- rows are never dropped silently."""
    EXCLUSION_LEDGER.append({"step": step, "reason": reason, "rows_before": int(before), "rows_removed": int(before - after), "rows_after": int(after)})
    print(f"[exclusion] {step}: {reason}: removed {before - after:,} of {before:,} -> {after:,} remain")


def bootstrap_ci(y_true, y_pred, metric, n_boot=500, seed=0, alpha=0.05, stratify=None):
    """Percentile bootstrap CI for metric(y_true, y_pred), resampling test rows (optionally stratified by a label vector)."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    idx_by = [np.where(np.asarray(stratify) == k)[0] for k in np.unique(stratify)] if stratify is not None else None
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in idx_by]) if idx_by else rng.integers(0, n, n)
        try:
            vals.append(metric(y_true[idx], y_pred[idx]))
        except ValueError:
            continue
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

In [ ]:
# ======================= description-derived features (no fitting; safe to compute before splitting) =======================
FLAG_PATTERNS = {
    "kw_rce": r"remote code execution|execute arbitrary (?:code|commands?)|arbitrary code|code execution|command injection|os command",
    "kw_sqli": r"sql injection|\bsqli\b",
    "kw_xss": r"cross[- ]site scripting|\bxss\b",
    "kw_memory": r"buffer overflow|out[- ]of[- ]bounds|heap[- ]based|stack[- ]based|use[- ]after[- ]free|memory corruption",
    "kw_auth_bypass": r"authentication bypass|bypass authentication|improper authentication|missing authentication",
    "kw_privesc": r"privilege escalation|escalat\w+ privileges|gain privileges|elevat\w+ privileges",
    "kw_dos": r"denial of service|\bdos\b|infinite loop|resource exhaustion",
    "kw_traversal": r"path traversal|directory traversal",
    "kw_csrf": r"cross[- ]site request forgery|\bcsrf\b",
    "kw_infoleak": r"information disclosure|sensitive information|information leak",
    "kw_remote": r"\bremote(?:ly)?\b|over the network",
    "kw_local": r"\blocal(?:ly)?\b",
    "kw_unauth": r"unauthenticated|without authentication|not require authentication|no authentication",
    "kw_auth": r"\bauthenticated\b|valid credentials|logged[- ]in|low[- ]privileged",
    "kw_user_interaction": r"user interaction|convince|trick|if a user|victim|\bclick",
    "kw_exploit_public": r"exploit (?:is|has been|code)[^.]{0,30}(?:public|disclosed|available|released)|publicly (?:exploit|disclosed)|proof[- ]of[- ]concept",
}
FLAG_COLUMNS = list(FLAG_PATTERNS)


def add_text_features(df):
    """Adds desc_length, desc_words and kw_* flags from description_en. Pure row-wise functions of the text -> no leakage."""
    d = df.copy()
    text = d["description_en"].fillna("")
    low = text.str.lower()
    d["desc_length"] = text.str.len().astype("int32")
    d["desc_words"] = text.str.split().str.len().astype("int32")
    for name, pat in FLAG_PATTERNS.items():
        d[name] = low.str.contains(pat, regex=True).astype("int8")
    return d


def normalize_description(s):
    """Normalisation used for near-duplicate detection: lower-case, digits -> '#', collapse whitespace."""
    return s.fillna("").str.lower().str.replace(r"\d+", "#", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

## 5. Dataset discovery and loading

In [ ]:
df_raw, DATA_META = load_dataset(need_exploit_signals=True)
HORIZON = pd.Timedelta(days=CONFIG["LABEL_HORIZON_DAYS"])
if CONFIG["FAST_DEV"]:
    pos_mask = df_raw["kev_dateadded"].notna()
    df_raw = pd.concat([df_raw[pos_mask], df_raw[~pos_mask].sample(CONFIG["FAST_DEV_NEGATIVES"], random_state=CONFIG["SEED"])]).reset_index(drop=True)
    print(f"!! FAST_DEV: all KEV rows + {CONFIG['FAST_DEV_NEGATIVES']:,} random non-KEV rows (prevalence is distorted). Do NOT report results from this run.")
DATA_META["fast_dev"] = CONFIG["FAST_DEV"]
print("\n================ SETUP SUMMARY ================")
print(f"environment        : {ENVIRONMENT}")
print(f"python / libs      : {DEPENDENCY_VERSIONS['python']} | numpy {DEPENDENCY_VERSIONS['numpy']} | pandas {DEPENDENCY_VERSIONS['pandas']} | sklearn {DEPENDENCY_VERSIONS['sklearn']}")
print(f"data source        : {DATA_META['source_kind']} -- {DATA_META['source_description']}")
for f in DATA_META["files"]:
    print(f"  file             : {f['name']}  {f['bytes']/1e6:.1f} MB  sha256={f['sha256'][:16]}...")
print(f"dataset fingerprint: {DATA_META['dataset_fingerprint_sha256']}")
print(f"rows x cols        : {df_raw.shape};  published {df_raw['published_date'].min().date()} -> {df_raw['published_date'].max().date()}")
print(f"label horizon      : {CONFIG['LABEL_HORIZON_DAYS']} days after publication; output dir {OUTPUT_DIR}")
print("================================================")

## 6. Schema and data-quality validation, target construction, and the data-sufficiency gate

In [ ]:
REQUIRED = ["cve_id", "published_date", "description_en", "vuln_status", "source_identifier", "primary_cwe_id", "reference_count", "cpe_match_count",
            "cvss_v31_vector", "cvss_v31_base_score", "cvss_v30_vector", "cvss_v30_base_score", "kev_is_known_exploited", "kev_dateadded", "epss_score", "epss_percentile"]
missing_cols = [c for c in REQUIRED if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Schema mismatch: missing {missing_cols}. Expected the afr1ste Kaggle schema or the official-feed route (ALLOW_NETWORK=True).")
print("schema OK;", df_raw.shape[1], "columns")
display((df_raw[REQUIRED].isna().mean() * 100).round(2).rename("pct_missing").to_frame().T)

df = df_raw.copy()
n = len(df); df = df[df["cve_id"].astype(str).str.match(CVE_ID_RE)]; ledger("cve_id", "malformed CVE identifier", n, len(df))
n = len(df); df = df.sort_values("last_modified_date").drop_duplicates("cve_id", keep="last"); ledger("duplicates", "duplicate CVE id (kept latest modified)", n, len(df))
n = len(df); df = df[df["vuln_status"].fillna("") != "Rejected"]; ledger("status", "NVD vuln_status == Rejected", n, len(df))
n = len(df); df = df[df["description_en"].fillna("").str.strip().ne("")]; ledger("description", "missing/empty description", n, len(df))
n = len(df); df = df[df["published_date"].notna()]; ledger("published", "missing/unparsable published date", n, len(df))

# KEV consistency checks
flag_mismatch = int((df["kev_is_known_exploited"].astype(bool) != df["kev_dateadded"].notna()).sum())
df["lag_days"] = (df["kev_dateadded"] - df["published_date"]).dt.total_seconds() / 86400
print(f"KEV flag vs dateAdded mismatches: {flag_mismatch}; CVEs in KEV (any time): {int(df['kev_dateadded'].notna().sum()):,}; "
      f"KEV added BEFORE NVD publication: {int((df['lag_days'] < 0).sum())} (counted as positive)")
assert flag_mismatch == 0, "kev_is_known_exploited disagrees with kev_dateadded -- data problem, stop."

# target construction + right-censoring
SNAPSHOT = pd.Timestamp(CONFIG["SNAPSHOT_DATE"]) if CONFIG["SNAPSHOT_DATE"] else max(df["published_date"].max(), df["kev_dateadded"].max())
df["y_kev_ever"] = df["kev_dateadded"].notna().astype(int)  # context only; NOT the target
df["y"] = (df["kev_dateadded"].notna() & (df["kev_dateadded"] <= df["published_date"] + HORIZON)).astype(int)
n = len(df); df = df[df["published_date"] <= SNAPSHOT - HORIZON]
ledger("right-censoring", f"published within {CONFIG['LABEL_HORIZON_DAYS']} days of snapshot {SNAPSHOT.date()}: label window not fully observed", n, len(df))
df = df.sort_values("published_date").reset_index(drop=True)
n_pos = int(df["y"].sum())
print(f"\neligible CVEs: {len(df):,}; positives (KEV within {CONFIG['LABEL_HORIZON_DAYS']}d): {n_pos:,} (prevalence {df['y'].mean():.4%}); "
      f"in KEV ever but outside window (counted NEGATIVE here): {int(((df['y_kev_ever'] == 1) & (df['y'] == 0)).sum()):,}")
display(pd.DataFrame(EXCLUSION_LEDGER))

# ---- data-sufficiency gate: refuse to train on an indefensible target ----
BLOCK_REASONS = []
if df["kev_dateadded"].notna().sum() == 0:
    BLOCK_REASONS.append("no KEV dates in the data -- attach the dataset with KEV columns or enable ALLOW_NETWORK for the official KEV file")
if n_pos < CONFIG["MIN_POSITIVES_TOTAL"]:
    BLOCK_REASONS.append(f"only {n_pos} positive labels (< MIN_POSITIVES_TOTAL={CONFIG['MIN_POSITIVES_TOTAL']}); too few to learn or evaluate")
if BLOCK_REASONS:
    write_json(OUTPUT_DIR / "blocked_report.json", {"status": "BLOCKED -- no model artifact produced", "reasons": BLOCK_REASONS, "rows": len(df), "positives": n_pos,
                                                    "what_is_missing": "KEV labels with dates covering enough CVEs that are old enough to observe the horizon"})
    raise RuntimeError("EXPERIMENT BLOCKED: " + "; ".join(BLOCK_REASONS) + f". Details written to {OUTPUT_DIR/'blocked_report.json'}. No model artifact was produced.")
print("data-sufficiency gate passed.")

## 7. Exploratory data analysis

In [ ]:
df["pub_year"] = df["published_date"].dt.year
df["pub_quarter"] = df["published_date"].dt.to_period("Q").astype(str)

# ---- TEXT-LEAKAGE GUARD ------------------------------------------------------------------------------
# NVD descriptions are CURRENT-state. For KEV-listed CVEs vendors/NVD frequently add sentences such as
# "... is aware of reports that this vulnerability has been exploited in the wild". That sentence IS the label.
# We therefore remove every sentence that states exploitation evidence, for training AND for inference.
EXPLOIT_STMT_PATTERN = (r"(?i)[^.!?]*\b(?:in the wild|actively exploit\w*|active exploitation|being exploited|(?:has|have|had) been exploited|known to be exploited|"
                        r"aware of (?:any )?reports?|zero[- ]day|0[- ]day|exploitation (?:of this|has been|was|is) (?:observed|detected|reported|occurring))\b[^.!?]*[.!?]?")


def scrub_exploitation_statements(s):
    return s.fillna("").str.replace(EXPLOIT_STMT_PATTERN, " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()   # may be "" -> caller must handle


df["description_raw"] = df["description_en"]                      # kept ONLY for the leakage ablation; never a model input
df["had_exploit_statement"] = df["description_raw"].fillna("").str.contains(EXPLOIT_STMT_PATTERN, regex=True)
df["description_en"] = scrub_exploitation_statements(df["description_raw"])
leak_rate = df.groupby("y")["had_exploit_statement"].agg(["mean", "sum"]).rename(index={0: "negatives", 1: "positives"})
display(leak_rate.rename(columns={"mean": "share_with_exploitation_statement", "sum": "n"}))
print(f"{int(df['had_exploit_statement'].sum()):,} descriptions contained an exploitation statement and were scrubbed "
      f"({df.loc[df.y == 1, 'had_exploit_statement'].mean():.0%} of positives vs {df.loc[df.y == 0, 'had_exploit_statement'].mean():.2%} of negatives -- a direct label leak if left in).")
# A description that consists ONLY of an exploitation statement has nothing left to featurise; any placeholder would itself leak the label.
n = len(df); df = df[df["description_en"].str.len() > 0].reset_index(drop=True)
ledger("text-leakage guard", "description consisted solely of exploitation statement(s); nothing left after scrubbing (a placeholder would leak the label)", n, len(df))
df = add_text_features(df)

# CVSS v3.x features (v3.1 preferred, else v3.0); versions are never mixed with v2/v4.
CVSS3_METRICS = ["AV", "AC", "PR", "UI", "S", "C", "I", "A"]
vec3 = df["cvss_v31_vector"].where(df["cvss_v31_vector"].notna(), df["cvss_v30_vector"])
df["cvss3_score"] = pd.to_numeric(df["cvss_v31_base_score"].where(df["cvss_v31_base_score"].notna(), df["cvss_v30_base_score"]), errors="coerce")
df["has_cvss3"] = df["cvss3_score"].notna().astype(int)
met = pd.DataFrame([(parse_cvss_vector(s) or (None, {}))[1] for s in vec3], index=df.index).reindex(columns=CVSS3_METRICS)
for m in CVSS3_METRICS:
    df[f"c3_{m}"] = met[m].fillna("MISSING").astype(str)
df["cvss3_band"] = [cvss_severity(s, "v31") or "NO CVSS v3" for s in df["cvss3_score"]]

# 1) label distribution + prevalence over time
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = [int((df.y == 0).sum()), int(df.y.sum())]
axes[0].bar(["negative (y=0)", "positive (y=1)"], counts, color=["#7a9cc6", "#c0504d"]); axes[0].set_yscale("log")
for i, v in enumerate(counts):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")
axes[0].set(title=f"Label distribution: KEV within {CONFIG['LABEL_HORIZON_DAYS']} days (prevalence {df.y.mean():.3%})", ylabel="CVE count (log scale)")
yr = df.groupby("pub_year")["y"].agg(["mean", "sum", "size"])
axes[1].bar(yr.index.astype(str), yr["mean"] * 100, color="#c0504d")
for i, (_, r) in enumerate(yr.iterrows()):
    axes[1].text(i, r["mean"] * 100, f"{int(r['sum'])}/{int(r['size']):,}", ha="center", va="bottom", fontsize=7)
axes[1].set(title="Positive rate by publication year (positives / CVEs)", xlabel="publication year", ylabel="% of CVEs positive")
savefig("01_label_distribution_and_prevalence")

# 2) class imbalance in numbers + lag to KEV
kev_any = df[df.y_kev_ever == 1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].pie(counts, labels=[f"negative\n{(df.y == 0).mean():.2%}", f"positive\n{df.y.mean():.2%}"], colors=["#7a9cc6", "#c0504d"], startangle=90)
axes[0].set_title(f"Class imbalance: {int(counts[0] / max(counts[1], 1)):,}:1 negative:positive")
axes[1].hist(kev_any["lag_days"].clip(-400, 1500), bins=60, color="#8a5a9e"); axes[1].axvline(CONFIG["LABEL_HORIZON_DAYS"], color="r", ls="--", label=f"horizon = {CONFIG['LABEL_HORIZON_DAYS']} d"); axes[1].axvline(0, color="k", lw=0.8)
axes[1].set(title="Days from NVD publication to KEV addition (all KEV CVEs; clipped)\nleft of 0: added before NVD publication", xlabel="days", ylabel="CVE count"); axes[1].legend()
savefig("02_class_imbalance_and_kev_lag")

# 3) feature distributions by label
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
sns.histplot(data=df[df.has_cvss3 == 1], x="cvss3_score", hue="y", stat="density", common_norm=False, bins=30, ax=axes[0, 0]); axes[0, 0].set(title="CVSS v3.x base score by label (density)", xlabel="CVSS v3 base score")
for ax, m, ttl in [(axes[0, 1], "c3_AV", "Attack vector"), (axes[0, 2], "c3_PR", "Privileges required")]:
    (pd.crosstab(df[m], df["y"], normalize="columns") * 100).plot(kind="bar", ax=ax, color=["#7a9cc6", "#c0504d"]); ax.set(title=f"{ttl} (v3.x) share by label", ylabel="% within label", xlabel="")
df[df.y == 1]["primary_cwe_id"].fillna("(missing)").value_counts().head(10).iloc[::-1].plot(kind="barh", ax=axes[1, 0], color="#c0504d"); axes[1, 0].set(title="Top-10 primary CWE among positives", xlabel="positive CVEs")
df[df.y == 1]["source_identifier"].fillna("(missing)").value_counts().head(10).iloc[::-1].plot(kind="barh", ax=axes[1, 1], color="#c0504d"); axes[1, 1].set(title="Top-10 reporting CNA among positives", xlabel="positive CVEs")
(df.groupby("y")[FLAG_COLUMNS].mean().T * 100).sort_values(1).plot(kind="barh", ax=axes[1, 2], color=["#7a9cc6", "#c0504d"]); axes[1, 2].set(title="Description keyword flags: % of CVEs by label", xlabel="% within label")
savefig("03_feature_distributions_by_label")

# 4) missingness + correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
(df[["cvss3_score", "primary_cwe_id", "source_identifier", "reference_count", "cpe_match_count", "epss_score"]].isna().mean() * 100).sort_values().plot(kind="barh", ax=axes[0], color="#c0504d"); axes[0].set(title="Missing values (% of eligible CVEs)", xlabel="% missing")
num_eda = ["cvss3_score", "has_cvss3", "desc_length", "desc_words"] + FLAG_COLUMNS + ["y"]
corr = df[num_eda].corr(method="spearman")
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-0.5, vmax=0.5, ax=axes[1], square=True, cbar_kws={"label": "Spearman rho"}); axes[1].set(title="Spearman correlation: eligible numeric features and the label 'y'\n(EDA only; 'y' is never an input)")
savefig("04_missingness_and_correlation")
display(corr["y"].drop("y").sort_values(key=np.abs, ascending=False).head(8).to_frame("spearman_with_label"))

## 8. Target and feature definitions
**Information set (prediction time ≈ shortly after publication):** CVSS v3.x base metrics + score (v3.1 preferred, else v3.0; absent → `MISSING` / `has_cvss3=0`), reporting CNA, primary CWE, description-derived length and keyword flags (incl. `kw_exploit_public`), and — for one candidate — TF-IDF of the description.

**Explicitly excluded:** `kev_*` and `y_kev_ever` / `lag_days` (label-derived), **`epss_*`** (target-adjacent, hindsight), `vuln_status`, `last_modified_date` (post-publication), `reference_count`, `cpe_match_count` (accumulate after publication and rise when exploit write-ups appear), **all CVSS v2 / v4.0 fields** (v2 vanished for new CVEs in mid-2022 → non-stationary), `published_date`/year (split key only).
A `leaky_demo` feature set *adds* reference/CPE counts and current EPSS purely to measure how misleading such fields would be (§14). It is never a candidate.
**Text-leakage guard:** current NVD descriptions of KEV-listed CVEs often contain sentences like "… aware of reports of exploitation in the wild". Those sentences are the label in prose, so they are **removed from `description_en`** (`scrub_exploitation_statements`) before any feature is computed, and again at inference. A second ablation (`leaky_text_demo`) keeps them to quantify the inflation.

In [ ]:
CVSS_CAT = [f"c3_{m}" for m in CVSS3_METRICS]
NUM = ["cvss3_score", "has_cvss3", "desc_length", "desc_words"] + FLAG_COLUMNS
CAT = CVSS_CAT + ["source_identifier", "primary_cwe_id"]
TEXT_COL = "description_en"
FEATURE_SETS = {
    "core": dict(text=False, cat=CAT, num=NUM),
    "core_text": dict(text=True, cat=CAT, num=NUM),
    "leaky_demo": dict(text=False, cat=CAT, num=NUM + ["reference_count", "cpe_match_count", "epss_score", "epss_percentile"]),  # NEVER a candidate
    "leaky_text_demo": dict(text=True, text_col="description_raw", cat=CAT, num=NUM),   # UNSCRUBBED descriptions: NEVER a candidate
}
FORBIDDEN_RE = re.compile(r"epss|kev|^y$|^y_|lag_days|vuln_status|last_modified|published|cvss_v2|cvss_v40|description_raw|had_exploit", re.I)


def feature_columns(spec):
    return ([spec.get("text_col", TEXT_COL)] if spec["text"] else []) + spec["cat"] + spec["num"]


for name, spec in FEATURE_SETS.items():
    bad = [c for c in feature_columns(spec) if FORBIDDEN_RE.search(c)]
    if name.startswith("leaky"):
        print(f"feature set {name:15s}: contains forbidden-by-design columns {bad} (ablation only)")
    else:
        assert not bad, f"LEAKAGE: feature set {name} contains {bad}"
        print(f"feature set {name:11s}: {len(feature_columns(spec)):3d} input columns (leakage name-audit passed)")

## 9. Leakage analysis

In [ ]:
leakage_table = pd.DataFrame([
    ("kev_*, y_kev_ever, lag_days", "label / label-derived", "EXCLUDED", "Directly encode the target"),
    ("epss_score / epss_percentile (current)", "model output formed AFTER exploitation evidence; KEV is among EPSS's own signals", "EXCLUDED (reference comparator + leaky_demo only)", "Would make this 'approximating EPSS', and leaks the future"),
    ("reference_count, cpe_match_count", "accumulate after publication (exploit write-ups add references)", "EXCLUDED (leaky_demo only)", "Mild-to-strong leak; quantified in §14"),
    ("description sentences stating exploitation ('exploited in the wild', 'aware of reports'...)", "current-state text edited AFTER exploitation became known -- the label written in prose", "SCRUBBED (removed) before any feature is computed; unscrubbed text only in `leaky_text_demo`", "Found by inspecting the model's top terms; quantified in §14"),
    ("vuln_status, last_modified_date", "post-publication processing state", "EXCLUDED", "Analysis outcome known only later"),
    ("cvss v2 / v4.0 fields", "version availability varies with time", "EXCLUDED", "Non-stationary under a chronological split; versions not mixed"),
    ("cvss v3.x metrics & score", "assigned by CNA/NVD shortly after publication", "INCLUDED (caveat)", "Current record may reflect later re-scoring"),
    ("description-derived flags/length, CNA, CWE", "available at/near publication", "INCLUDED (caveat)", "Descriptions/CWE may be edited later"),
    ("published_date", "split key", "NOT a feature", "Year is non-stationary"),
], columns=["column group", "temporal status", "decision", "reason"])
display(leakage_table)
dup_rate = normalize_description(df["description_en"]).duplicated(keep=False).mean()
print(f"duplicate CVE ids: {int(df['cve_id'].duplicated().sum())}; rows sharing a normalised description with another CVE: {dup_rate:.1%} (templated products inflate random splits; we split by time)")

## 10. Train / validation / test strategy — chronological **with label-horizon embargo**
A CVE's label is only knowable `LABEL_HORIZON_DAYS` after its publication. To avoid training on labels that "peek" past the start of the next period, each split boundary `t` is preceded by an **embargo**: the earlier split keeps only CVEs published ≤ `t − horizon`; CVEs in between are dropped (and counted).
train: published ≤ `t1 − H` · validation: `t1 < published ≤ t2 − H` · test: published > `t2`. Validation is used for model selection, probability calibration and threshold choice; **test is evaluated once**.

In [ ]:
t1, t2 = df["published_date"].quantile(CONFIG["TIME_SPLIT_QUANTILES"][0]), df["published_date"].quantile(CONFIG["TIME_SPLIT_QUANTILES"][1])
pub = df["published_date"]
m_tr, m_va, m_te = pub <= t1 - HORIZON, (pub > t1) & (pub <= t2 - HORIZON), pub > t2
tr_idx, va_idx, te_idx = np.where(m_tr)[0], np.where(m_va)[0], np.where(m_te)[0]
n_embargo = len(df) - len(tr_idx) - len(va_idx) - len(te_idx)
EXCLUSION_LEDGER.append({"step": "embargo", "reason": f"within {CONFIG['LABEL_HORIZON_DAYS']} days before a split boundary (prevents label look-ahead)", "rows_before": len(df), "rows_removed": n_embargo, "rows_after": len(df) - n_embargo})
split_table = pd.DataFrame([{"split": nm, "rows": len(ix), "positives": int(df.y.values[ix].sum()), "prevalence_%": round(100 * df.y.values[ix].mean(), 3),
                             "from": df.published_date.iloc[ix].min().date(), "to": df.published_date.iloc[ix].max().date()} for nm, ix in [("train", tr_idx), ("validation", va_idx), ("test", te_idx)]])
display(split_table)
print(f"embargoed rows dropped: {n_embargo:,}")
short = [r["split"] for _, r in split_table.iterrows() if r["positives"] < CONFIG["MIN_POSITIVES_PER_SPLIT"]]
if short:
    write_json(OUTPUT_DIR / "blocked_report.json", {"status": "BLOCKED -- no model artifact produced", "reasons": [f"split(s) {short} have < {CONFIG['MIN_POSITIVES_PER_SPLIT']} positives"], "split_table": split_table.to_dict(orient="records")})
    raise RuntimeError(f"EXPERIMENT BLOCKED: split(s) {short} have fewer than {CONFIG['MIN_POSITIVES_PER_SPLIT']} positives -- metrics would be meaningless. "
                       "Try a shorter LABEL_HORIZON_DAYS, different TIME_SPLIT_QUANTILES, or more years of data (NVD_YEARS).")

## 11. Baselines and metric helpers
Why **accuracy is useless here:** with ~1.5 % positives (and ~0.4 % in the full data), a classifier that never predicts "exploited" scores ≈ 99 % accuracy yet finds nothing. We therefore report **PR-AUC (average precision)** — whose chance level equals the prevalence — ROC-AUC, Brier score, calibration, and precision/recall at thresholds and review budgets (top-k).
Baselines: (1) `DummyClassifier(prior)`; (2) **rank by CVSS v3 score alone** (a fair, simple heuristic); (3) *reference only, hindsight-contaminated*: rank by **current EPSS** — EPSS already ingests post-publication exploitation evidence (including KEV-like signals), so it is **not** a fair out-of-sample forecaster and is shown only to calibrate expectations.

In [ ]:
from scipy import sparse
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

if tuple(int(x) for x in sklearn.__version__.split(".")[:2]) < (1, 3):
    raise ImportError(f"scikit-learn >= 1.3 required (found {sklearn.__version__}).")
N_JOBS, SEED = CONFIG["N_JOBS"], CONFIG["SEED"]
Y = df["y"].values
y_tr, y_va, y_te = Y[tr_idx], Y[va_idx], Y[te_idx]
PREV_TE = y_te.mean()


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); w = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - w) / d, (c + w) / d)


def rank_metrics(y, s, ks=CONFIG["TOPK_FRACTIONS"]):
    """Threshold-free metrics + top-k review-budget metrics."""
    out = {"pr_auc": float(average_precision_score(y, s)), "roc_auc": float(roc_auc_score(y, s)), "prevalence": float(np.mean(y))}
    order = np.argsort(-np.asarray(s), kind="stable"); ys = np.asarray(y)[order]
    for k in ks:
        m = max(int(round(k * len(y))), 1)
        out[f"recall@{int(k*100)}%"] = float(ys[:m].sum() / max(ys.sum(), 1)); out[f"precision@{int(k*100)}%"] = float(ys[:m].mean())
        out[f"lift@{int(k*100)}%"] = float(ys[:m].mean() / max(np.mean(y), 1e-12))
    return out


def at_threshold(y, p, thr):
    pred = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"threshold": float(thr), "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn), "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),
            "f1": 2 * tp / max(2 * tp + fp + fn, 1), "flagged_fraction": float(pred.mean()), "recall_ci95": wilson(tp, tp + fn), "precision_ci95": wilson(tp, tp + fp)}


# ---- non-learned baselines on the test set ----
cvss_rank = np.nan_to_num(df["cvss3_score"].values[te_idx], nan=-1.0)
BASELINES = {"dummy_prior": rank_metrics(y_te, np.full(len(y_te), y_tr.mean()) + np.random.default_rng(SEED).normal(0, 1e-9, len(y_te))),
             "rank_by_cvss3_score": rank_metrics(y_te, cvss_rank),
             "EPSS_current (hindsight-contaminated reference)": rank_metrics(y_te, np.nan_to_num(df["epss_score"].values[te_idx], nan=0.0))}
display(pd.DataFrame(BASELINES).T.round(4))

## 12. Candidate models, tuning (validation only) and held-out scores
All preprocessing lives in the pipeline and is **fit on the training rows only**. Imbalance handling = class weights (LR, RF) / none for HGB (probabilities are calibrated afterwards); **no resampling**, and nothing from validation/test is used for fitting except where stated (calibration and threshold use validation only).
Candidates: `logreg` (core), `logreg_text` (core + TF-IDF), `random_forest`, `hist_gradient_boosting`. Selection metric = **validation PR-AUC**.

In [ ]:
def make_prep(spec, family):
    parts = []
    if spec["text"]:
        parts.append(("text", TfidfVectorizer(max_features=CONFIG["TFIDF_MAX_FEATURES"], ngram_range=(1, 2), min_df=5, sublinear_tf=True, dtype=np.float32), spec.get("text_col", TEXT_COL)))
    parts.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=(family == "linear")), spec["cat"]))
    if family == "linear":
        parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), spec["num"]))
    else:
        parts.append(("num", SimpleImputer(strategy="constant", fill_value=-1.0), spec["num"]))
    return ColumnTransformer(parts, sparse_threshold=0.3 if family == "linear" else 0.0)


def estimator(kind, **kw):
    return {"dummy_prior": lambda: DummyClassifier(strategy="prior"),
            "logreg": lambda: LogisticRegression(C=kw.get("C", 0.1), class_weight="balanced", max_iter=500, random_state=SEED),
            "logreg_text": lambda: LogisticRegression(C=kw.get("C", 0.1), class_weight="balanced", max_iter=500, random_state=SEED),
            "random_forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=kw.get("min_samples_leaf", 10), max_features="sqrt", class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=SEED),
            "hist_gradient_boosting": lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=kw.get("learning_rate", 0.05), max_leaf_nodes=kw.get("max_leaf_nodes", 15),
                                                                             min_samples_leaf=50, l2_regularization=1.0, early_stopping=False, random_state=SEED)}[kind]()


KIND_SPEC = {"dummy_prior": ("core", "linear"), "logreg": ("core", "linear"), "logreg_text": ("core_text", "linear"),
             "random_forest": ("core", "tree"), "hist_gradient_boosting": ("core", "tree")}
GRIDS = {"logreg": [{"C": c} for c in (0.03, 0.1, 0.3, 1.0, 3.0, 10.0)], "logreg_text": [{"C": c} for c in (0.1, 0.3, 1.0, 3.0, 10.0)],
         "random_forest": [{"min_samples_leaf": m} for m in (5, 20)],
         "hist_gradient_boosting": [{"learning_rate": lr, "max_leaf_nodes": ml} for lr in (0.05, 0.1) for ml in (15, 31)]}

PREPS = {}
for key in sorted(set(KIND_SPEC.values())):
    spec_name, fam = key
    cols = feature_columns(FEATURE_SETS[spec_name])
    p = make_prep(FEATURE_SETS[spec_name], fam).fit(df.iloc[tr_idx][cols])
    PREPS[key] = dict(prep=p, cols=cols, train=p.transform(df.iloc[tr_idx][cols]), val=p.transform(df.iloc[va_idx][cols]), test=p.transform(df.iloc[te_idx][cols]))
    print(f"preprocessing [{spec_name}/{fam}] fit on {len(tr_idx):,} training rows -> {PREPS[key]['train'].shape}")

TUNING_LOG, FITTED, BEST_PARAMS, RAW_VAL, RAW_TEST = [], {}, {}, {}, {}
for kind in KIND_SPEC:
    M = PREPS[KIND_SPEC[kind]]
    best = None
    for params in GRIDS.get(kind, [{}]):
        t0 = time.time()
        est = estimator(kind, **params).fit(M["train"], y_tr)
        s_val_ = est.predict_proba(M["val"])[:, 1]
        ap = average_precision_score(y_va, s_val_)
        TUNING_LOG.append({"model": kind, "params": json.dumps(params), "val_pr_auc": ap, "seconds": round(time.time() - t0, 1)})
        if best is None or ap > best[0]:
            best = (ap, params, est, s_val_)
    FITTED[kind], BEST_PARAMS[kind], RAW_VAL[kind] = best[2], best[1], best[3]
    RAW_TEST[kind] = best[2].predict_proba(M["test"])[:, 1]
    print(f"{kind:24s} best {best[1]}  val PR-AUC {best[0]:.4f}")

ROWS = []
for kind in FITTED:
    rm_v, rm_t = rank_metrics(y_va, RAW_VAL[kind]), rank_metrics(y_te, RAW_TEST[kind])
    lo, hi = bootstrap_ci(y_te, RAW_TEST[kind], average_precision_score, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED, stratify=y_te)
    rlo, rhi = bootstrap_ci(y_te, RAW_TEST[kind], roc_auc_score, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED, stratify=y_te)
    ROWS.append({"model": kind, "params": json.dumps(BEST_PARAMS[kind]), "val_pr_auc": rm_v["pr_auc"], "val_roc_auc": rm_v["roc_auc"], "test_pr_auc": rm_t["pr_auc"], "test_pr_auc_ci_low": lo, "test_pr_auc_ci_high": hi,
                 "test_roc_auc": rm_t["roc_auc"], "test_roc_auc_ci_low": rlo, "test_roc_auc_ci_high": rhi, "test_brier_raw": brier_score_loss(y_te, RAW_TEST[kind]),
                 **{k: v for k, v in rm_t.items() if k.startswith(("recall@", "precision@", "lift@"))}})
model_cmp = pd.DataFrame(ROWS).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
display(model_cmp[["model", "val_pr_auc", "test_pr_auc", "test_pr_auc_ci_low", "test_pr_auc_ci_high", "test_roc_auc", "test_brier_raw", "recall@5%", "precision@5%", "lift@5%"]].round(4))
cand = model_cmp[model_cmp.model != "dummy_prior"]
SELECTED = cand.iloc[0]["model"]  # selection on VALIDATION PR-AUC only
dummy_val = float(model_cmp.loc[model_cmp.model == "dummy_prior", "val_pr_auc"].iloc[0])
assert float(cand.iloc[0]["val_pr_auc"]) > 2 * dummy_val, "Best candidate does not clearly beat the prevalence baseline on validation (PR-AUC < 2x chance) -- do not export a model."
print(f"\nselected model (validation PR-AUC): {SELECTED}  | test chance level (prevalence) = {PREV_TE:.4f}")

## 13. Probability calibration (fitted on **validation only**) and threshold selection
The selected model is trained on the training period only; its raw scores on the **validation** period are used to fit a calibrator (Platt sigmoid or isotonic). The method is chosen by 5-fold CV Brier score *within the validation set*; the final calibrator is fit on all validation scores. **The test set is never used** for fitting the calibrator, choosing the method, or choosing the threshold. (Consequence: the exported model is "train-period model + validation-fitted calibrator", not refit on validation.)

In [ ]:
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))


def fit_calibrator(method, s, y):
    if method == "platt":
        return LogisticRegression(C=1e6, max_iter=1000).fit(logit(s).reshape(-1, 1), y)
    return IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(s, y)


def apply_calibrator(method, cal, s):
    return cal.predict_proba(logit(np.asarray(s)).reshape(-1, 1))[:, 1] if method == "platt" else cal.predict(np.asarray(s))


s_val, s_te = RAW_VAL[SELECTED], RAW_TEST[SELECTED]
cv_brier = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for method in ("platt", "isotonic"):
    oof = np.zeros(len(y_va))
    for a, b in skf.split(s_val, y_va):
        oof[b] = apply_calibrator(method, fit_calibrator(method, s_val[a], y_va[a]), s_val[b])
    cv_brier[method] = brier_score_loss(y_va, oof)
CAL_METHOD = min(cv_brier, key=cv_brier.get)
calibrator = fit_calibrator(CAL_METHOD, s_val, y_va)
p_va, p_te = apply_calibrator(CAL_METHOD, calibrator, s_val), apply_calibrator(CAL_METHOD, calibrator, s_te)
print(f"within-validation 5-fold CV Brier: {cv_brier}  -> chosen: {CAL_METHOD}")
BRIER = {"constant_prevalence_baseline": brier_score_loss(y_te, np.full(len(y_te), y_tr.mean())), "raw": brier_score_loss(y_te, s_te), "calibrated": brier_score_loss(y_te, p_te)}
print("TEST Brier:", {k: round(v, 5) for k, v in BRIER.items()}, "| Brier skill vs prevalence baseline (calibrated):", round(1 - BRIER["calibrated"] / BRIER["constant_prevalence_baseline"], 3))

# threshold selection on VALIDATION calibrated probabilities
prec, rec, thr = precision_recall_curve(y_va, p_va)
f1s = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
THR_F1 = float(thr[np.argmax(f1s)])
ok = np.where(rec[:-1] >= CONFIG["RECALL_TARGET"])[0]
THR_RECALL = float(thr[ok[np.argmax(prec[:-1][ok])]]) if len(ok) else float(thr[np.argmax(f1s)])
THRESHOLD = THR_RECALL if CONFIG["THRESHOLD_POLICY"] == "recall_target" else THR_F1
print(f"validation-selected thresholds (calibrated prob): max-F1 = {THR_F1:.4f}; recall>={CONFIG['RECALL_TARGET']:.0%} = {THR_RECALL:.4f}; policy '{CONFIG['THRESHOLD_POLICY']}' -> {THRESHOLD:.4f}")
_drop = ("recall_ci95", "precision_ci95")
THR_TABLE = pd.DataFrame([{"policy": nm, **{k: v for k, v in at_threshold(y_te, p_te, t).items() if k not in _drop}, "split": "TEST"} for nm, t in
                          [("max_f1 (chosen on val)", THR_F1), (f"recall>={CONFIG['RECALL_TARGET']:.0%} (chosen on val)", THR_RECALL), ("selected policy", THRESHOLD)]]
                         + [{"policy": f"top-{int(k*100)}% review budget", **{kk: vv for kk, vv in at_threshold(y_te, p_te, np.quantile(p_te, 1 - k)).items() if kk not in _drop}, "split": "TEST"} for k in CONFIG["TOPK_FRACTIONS"]])
display(THR_TABLE.round(4))
sel = at_threshold(y_te, p_te, THRESHOLD)
print(f"At the selected threshold on TEST: recall {sel['recall']:.2f} (95% Wilson {sel['recall_ci95'][0]:.2f}-{sel['recall_ci95'][1]:.2f}), precision {sel['precision']:.3f} "
      f"({sel['precision_ci95'][0]:.3f}-{sel['precision_ci95'][1]:.3f}); flags {sel['flagged_fraction']:.1%} of CVEs ({sel['tp']} TP, {sel['fp']} FP, {sel['fn']} FN).")

## 14. Held-out evaluation summary, leakage ablation and performance over time
* **Leakage ablation:** the same HGB hyper-parameters on `leaky_demo` (adds reference/CPE counts and *current EPSS*) — shows how misleading such fields are. Never a candidate.
* **Over time:** PR-AUC and recall@5 % per calendar quarter of the test period (quarters with < 5 positives are flagged and not interpretable).

In [ ]:
rank_sel = rank_metrics(y_te, p_te)
ci_sel = {"pr_auc": bootstrap_ci(y_te, p_te, average_precision_score, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED, stratify=y_te), "roc_auc": bootstrap_ci(y_te, p_te, roc_auc_score, n_boot=CONFIG["N_BOOTSTRAP"], seed=SEED, stratify=y_te)}
print(f"SELECTED {SELECTED} TEST: PR-AUC {rank_sel['pr_auc']:.3f} {tuple(round(x, 3) for x in ci_sel['pr_auc'])} (chance {PREV_TE:.4f}), ROC-AUC {rank_sel['roc_auc']:.3f} {tuple(round(x, 3) for x in ci_sel['roc_auc'])}")

# leakage ablation (HGB)
spec_l = FEATURE_SETS["leaky_demo"]; cols_l = feature_columns(spec_l)
pl = make_prep(spec_l, "tree").fit(df.iloc[tr_idx][cols_l])
hgb_l = estimator("hist_gradient_boosting", **BEST_PARAMS["hist_gradient_boosting"]).fit(pl.transform(df.iloc[tr_idx][cols_l]), y_tr)
leaky_te = hgb_l.predict_proba(pl.transform(df.iloc[te_idx][cols_l]))[:, 1]
ABLATION = pd.DataFrame([{"feature_set": "core (honest)", "model": "hist_gradient_boosting", "test_pr_auc": float(model_cmp.loc[model_cmp.model == "hist_gradient_boosting", "test_pr_auc"].iloc[0]),
                          "test_roc_auc": float(model_cmp.loc[model_cmp.model == "hist_gradient_boosting", "test_roc_auc"].iloc[0])},
                         {"feature_set": "leaky_demo (+refs, CPE counts, current EPSS)  <-- LEAKED", "model": "hist_gradient_boosting", "test_pr_auc": average_precision_score(y_te, leaky_te), "test_roc_auc": roc_auc_score(y_te, leaky_te)}])
# text-leakage ablation: the same logistic-regression text model on UNSCRUBBED descriptions (exploitation sentences left in)
spec_t = FEATURE_SETS["leaky_text_demo"]; cols_t = feature_columns(spec_t)
pt_ = make_prep(spec_t, "linear").fit(df.iloc[tr_idx][cols_t])
lr_t = estimator("logreg_text", **BEST_PARAMS["logreg_text"]).fit(pt_.transform(df.iloc[tr_idx][cols_t]), y_tr)
leaky_text_te = lr_t.predict_proba(pt_.transform(df.iloc[te_idx][cols_t]))[:, 1]
ABLATION = pd.concat([ABLATION, pd.DataFrame([
    {"feature_set": "core_text (honest, scrubbed descriptions)", "model": "logreg_text", "test_pr_auc": float(model_cmp.loc[model_cmp.model == "logreg_text", "test_pr_auc"].iloc[0]), "test_roc_auc": float(model_cmp.loc[model_cmp.model == "logreg_text", "test_roc_auc"].iloc[0])},
    {"feature_set": "core_text on UNSCRUBBED descriptions  <-- LEAKED", "model": "logreg_text", "test_pr_auc": average_precision_score(y_te, leaky_text_te), "test_roc_auc": roc_auc_score(y_te, leaky_text_te)}])], ignore_index=True)
display(ABLATION.round(4))

# over time (test quarters)
te_df = df.iloc[te_idx].assign(p=p_te)
QROWS = []
for q, g in te_df.groupby("pub_quarter"):
    npos = int(g.y.sum())
    QROWS.append({"quarter": q, "n": len(g), "positives": npos, "pr_auc": average_precision_score(g.y, g.p) if npos >= 5 else np.nan,
                  "recall@5%": rank_metrics(g.y.values, g.p.values, ks=(0.05,))["recall@5%"] if npos >= 1 else np.nan, "interpretable": npos >= 5})
quarter_df = pd.DataFrame(QROWS)
display(quarter_df.round(3))

## 15. Required visualisations
Curves use the **calibrated** probabilities of the selected model on the **test** split unless a panel says otherwise. Baselines are drawn on the same axes; the EPSS curve is labelled as a hindsight-contaminated reference.

In [ ]:
BASE_SCORES = {"CVSS v3 score only": cvss_rank, "EPSS current (hindsight ref.)": np.nan_to_num(df["epss_score"].values[te_idx], nan=0.0)}
# 1) ROC
fig, ax = plt.subplots(figsize=(6.2, 5.4))
fpr, tpr, _ = roc_curve(y_te, p_te); ax.plot(fpr, tpr, lw=2.2, label=f"{SELECTED} (AUC {rank_sel['roc_auc']:.3f})")
for nm, sc in BASE_SCORES.items():
    f_, t_, _ = roc_curve(y_te, sc); ax.plot(f_, t_, lw=1.2, ls="--", label=f"{nm} (AUC {roc_auc_score(y_te, sc):.3f})")
ax.plot([0, 1], [0, 1], "k:", lw=1, label="chance"); ax.set(xlabel="false-positive rate", ylabel="true-positive rate (recall)", title="ROC curve, chronological test\n(ROC looks optimistic under extreme imbalance -- see PR curve)"); ax.legend(fontsize=8)
savefig("E1_roc_curve")
# 2) PR curve
fig, ax = plt.subplots(figsize=(6.2, 5.4))
pr_, rc_, _ = precision_recall_curve(y_te, p_te); ax.plot(rc_, pr_, lw=2.2, label=f"{SELECTED} (AP {rank_sel['pr_auc']:.3f}; 95% CI {ci_sel['pr_auc'][0]:.2f}-{ci_sel['pr_auc'][1]:.2f})")
for nm, sc in BASE_SCORES.items():
    p2, r2, _ = precision_recall_curve(y_te, sc); ax.plot(r2, p2, lw=1.2, ls="--", label=f"{nm} (AP {average_precision_score(y_te, sc):.3f})")
ax.axhline(PREV_TE, color="k", ls=":", label=f"chance = prevalence {PREV_TE:.3f}"); ax.scatter([sel["recall"]], [sel["precision"]], color="red", zorder=5, label=f"selected threshold ({THRESHOLD:.3f})")
ax.set(xlabel="recall", ylabel="precision", ylim=(0, 1.02), title="Precision-recall curve, chronological test"); ax.legend(fontsize=8)
savefig("E2_precision_recall_curve")
# 3) calibration (reliability) + probability histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
for nm, pp, col in [("raw model score", s_te, "#d08a3b"), (f"calibrated ({CAL_METHOD}, fitted on validation)", p_te, "#3b6ea5")]:
    fr, mp = calibration_curve(y_te, pp, n_bins=8, strategy="quantile"); axes[0].plot(mp, fr, "o-", color=col, label=nm)
mx = max(p_te.max(), 0.05); axes[0].plot([0, mx], [0, mx], "k:", label="perfect calibration")
axes[0].set(xlabel="mean predicted probability (quantile bins)", ylabel="observed fraction of positives", title="Reliability curve, test\n(8 quantile bins; sparse positives => wide uncertainty)"); axes[0].legend(fontsize=8)
axes[1].hist(p_te, bins=np.logspace(np.log10(max(p_te.min(), 1e-5)), 0, 40), color="#3b6ea5"); axes[1].set_xscale("log"); axes[1].set(xlabel="calibrated probability (log scale)", ylabel="CVE count", title="Predicted-probability distribution (test)")
savefig("E3_calibration_and_probability_distribution")
# 4) confusion matrix at the selected threshold
cmx = np.array([[sel["tn"], sel["fp"]], [sel["fn"], sel["tp"]]])
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
sns.heatmap(cmx, annot=True, fmt="d", cmap="Blues", xticklabels=["not flagged", "flagged"], yticklabels=["y=0", "y=1 (KEV in window)"], ax=axes[0]); axes[0].set(title=f"Confusion matrix @ threshold {THRESHOLD:.3f} (test)", xlabel="model decision", ylabel="label")
sns.heatmap(cmx / cmx.sum(1, keepdims=True), annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1, xticklabels=["not flagged", "flagged"], yticklabels=["y=0", "y=1"], ax=axes[1]); axes[1].set(title="Row-normalised (rates)", xlabel="model decision", ylabel="label")
savefig("E4_confusion_matrix_at_threshold")
# 5) precision / recall / F1 versus threshold
ts = np.unique(np.quantile(p_te, np.linspace(0.5, 0.9995, 200)))
curve = pd.DataFrame([{k: v for k, v in at_threshold(y_te, p_te, t).items() if k in ("threshold", "precision", "recall", "f1", "flagged_fraction")} for t in ts])
fig, ax = plt.subplots(figsize=(7.4, 4.6))
for c, col in [("precision", "#c0504d"), ("recall", "#3b6ea5"), ("f1", "#5b9a5b")]:
    ax.plot(curve["threshold"], curve[c], label=c, color=col)
ax.axvline(THRESHOLD, color="k", ls="--", label=f"validation-selected ({THRESHOLD:.3f})"); ax.set_xscale("log")
ax.set(xlabel="decision threshold on calibrated probability (log)", ylabel="metric value (test)", title="Precision / recall / F1 versus threshold\n(curves on TEST for display; threshold chosen on VALIDATION)"); ax.legend()
savefig("E5_threshold_curves")
# 6) predicted probability by class
fig, ax = plt.subplots(figsize=(7.4, 4.2))
bins = np.logspace(np.log10(max(p_te.min(), 1e-5)), 0, 40)
ax.hist(p_te[y_te == 0], bins=bins, alpha=0.6, density=True, label="y=0", color="#7a9cc6"); ax.hist(p_te[y_te == 1], bins=bins, alpha=0.6, density=True, label="y=1", color="#c0504d")
ax.set_xscale("log"); ax.axvline(THRESHOLD, color="k", ls="--"); ax.set(xlabel="calibrated probability (log)", ylabel="density", title="Predicted probability by true label (test)"); ax.legend()
savefig("E6_probability_by_class")
# 7) model comparison + leakage ablation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
m_ = model_cmp.sort_values("test_pr_auc")
axes[0].barh(m_["model"], m_["test_pr_auc"], xerr=[m_["test_pr_auc"] - m_["test_pr_auc_ci_low"], m_["test_pr_auc_ci_high"] - m_["test_pr_auc"]], color="#3b6ea5", capsize=3, label="test (95% bootstrap CI)")
axes[0].scatter(m_["val_pr_auc"], m_["model"], color="k", zorder=5, marker="D", label="validation (used for selection)"); axes[0].axvline(PREV_TE, color="r", ls=":", label="chance"); axes[0].legend(fontsize=8)
axes[0].set(title="PR-AUC by model (raw scores)", xlabel="average precision")
sns.barplot(data=ABLATION.assign(set=["HGB honest", "HGB LEAKED\n(+refs/CPE/EPSS)", "LR-text honest\n(scrubbed)", "LR-text LEAKED\n(raw text)"]), x="set", y="test_pr_auc", ax=axes[1], color="#d08a3b"); axes[1].set(title="Leakage ablations: honest vs leaked inputs", xlabel="", ylabel="test PR-AUC")
savefig("E7_model_comparison_and_leakage_ablation")
# 8) gain curve (top-k)  + performance over time
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
fr_ = np.linspace(0.001, 0.2, 100)
for nm, sc in [(SELECTED, p_te)] + list(BASE_SCORES.items()):
    o = np.argsort(-sc, kind="stable"); cs = np.cumsum(y_te[o]) / y_te.sum()
    axes[0].plot(fr_ * 100, [cs[max(int(f * len(y_te)) - 1, 0)] for f in fr_], label=nm, lw=2 if nm == SELECTED else 1.2, ls="-" if nm == SELECTED else "--")
axes[0].plot(fr_ * 100, fr_, "k:", label="random"); axes[0].set(xlabel="% of CVEs reviewed (highest score first)", ylabel="fraction of KEV-bound CVEs found (recall)", title="Cumulative gain / top-k recall, test"); axes[0].legend(fontsize=8)
qd = quarter_df.reset_index(drop=True)
axes[1].bar(qd["quarter"], qd["recall@5%"].fillna(0), color=np.where(qd["interpretable"], "#3b6ea5", "#bbbbbb"))
for i, r in qd.iterrows():
    axes[1].text(i, (r["recall@5%"] if pd.notna(r["recall@5%"]) else 0), f"pos={r['positives']}", ha="center", va="bottom", fontsize=8)
axes[1].set(title="Recall@5% review budget by test quarter (grey: < 5 positives)", xlabel="publication quarter", ylabel="recall@5%", ylim=(0, 1.05))
savefig("E8_gain_curve_and_performance_over_time")

## 16. Error and subgroup analysis
**False positives are not necessarily wrong:** a "negative" only means *not in KEV within the window*. We separate FPs that **were later added to KEV** (late additions → label-window artefact) from FPs with no KEV record (possible unreported exploitation **or** genuine over-prediction — indistinguishable with this data). **False negatives** are positives scored below the threshold; many were added to KEV *before or at* publication, i.e. exploited before any metadata existed.

In [ ]:
te_df = df.iloc[te_idx].assign(p=p_te)
te_df["pred"] = (te_df["p"] >= THRESHOLD).astype(int)
fp = te_df[(te_df.pred == 1) & (te_df.y == 0)].sort_values("p", ascending=False)
fn = te_df[(te_df.pred == 0) & (te_df.y == 1)].sort_values("p")
late = fp["y_kev_ever"].mean() if len(fp) else float("nan")
fn_pre = float((fn["lag_days"] <= 0).mean()) if len(fn) else float("nan")
pos_pre = float((te_df.loc[te_df.y == 1, "lag_days"] <= 0).mean())
print(f"False positives: {len(fp):,} (flagged but not in KEV within window); {late:.1%} of them were added to KEV LATER (outside the window); among the 50 highest-scored FPs: {fp.head(50)['y_kev_ever'].mean():.0%}.")
print(f"False negatives: {len(fn):,}; {fn_pre:.0%} were already in KEV at/before NVD publication (unpredictable from publication-time metadata). Share of ALL test positives listed at/before publication: {pos_pre:.0%}.")
cols_show = ["cve_id", "p", "cvss3_score", "primary_cwe_id", "source_identifier", "y_kev_ever", "lag_days", "description_en"]
fp_show, fn_show = fp.head(10)[cols_show].copy(), fn.head(10)[cols_show].copy()
for t in (fp_show, fn_show):
    t["description_en"] = t["description_en"].str.slice(0, 90)
print("\nHighest-scored false positives:"); display(fp_show.round(3))
print("Lowest-scored false negatives:"); display(fn_show.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].hist([fp["p"], te_df.loc[(te_df.pred == 1) & (te_df.y == 1), "p"]], bins=25, label=["false positives", "true positives"], color=["#d08a3b", "#5b9a5b"]); axes[0].set(title="Calibrated probability of flagged CVEs: FP vs TP", xlabel="calibrated probability", ylabel="CVE count"); axes[0].legend()
axes[1].hist(fn["lag_days"].clip(-400, 100), bins=30, color="#c0504d"); axes[1].axvline(0, color="k"); axes[1].set(title="False negatives: days from NVD publication to KEV addition (clipped)\nleft of 0 = listed in KEV before NVD publication", xlabel="days", ylabel="CVE count")
savefig("E9_false_positive_false_negative_analysis")

SUB = []
for dim, col, top in [("CVSS v3 band", "cvss3_band", None), ("reporting CNA (top 8 by test positives)", "source_identifier", 8), ("primary CWE (top 8 by test positives)", "primary_cwe_id", 8)]:
    g = te_df[col].fillna("(missing)")
    keep = te_df.assign(_g=g).groupby("_g")["y"].sum().sort_values(ascending=False).head(top).index if top else g.unique()
    for val in sorted(keep, key=str):
        s = te_df[g == val]; a = at_threshold(s.y.values, s.p.values, THRESHOLD)
        SUB.append({"dimension": dim, "group": str(val), "n": len(s), "positives": int(s.y.sum()), "recall": a["recall"] if s.y.sum() else np.nan, "precision": a["precision"] if a["tp"] + a["fp"] else np.nan,
                    "flagged_fraction": a["flagged_fraction"], "pr_auc": average_precision_score(s.y, s.p) if s.y.sum() >= 5 and s.y.nunique() > 1 else np.nan, "low_n_flag": int(s.y.sum()) < 10})
subgroup_df = pd.DataFrame(SUB)
display(subgroup_df.round(3))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, dim in zip(axes, subgroup_df.dimension.unique()):
    s = subgroup_df[subgroup_df.dimension == dim].reset_index(drop=True)
    ax.bar(s["group"].str.slice(0, 16), s["recall"].fillna(0), color=np.where(s["low_n_flag"], "#bbbbbb", "#3b6ea5"))
    for i, r in s.iterrows():
        ax.text(i, (r["recall"] if pd.notna(r["recall"]) else 0), f"pos={r['positives']}", ha="center", va="bottom", fontsize=7)
    ax.set(title=f"Recall @ threshold by {dim}\n(grey: < 10 positives)", ylabel="recall", ylim=(0, 1.1)); ax.tick_params(axis="x", rotation=45)
    ax.axhline(sel["recall"], color="r", ls="--", lw=1)
savefig("E10_subgroup_performance")

## 17. Model interpretation
Permutation importance = drop in **PR-AUC** on a validation sample (all positives + random negatives) when one input column is shuffled, using the train-only model. For the linear model, coefficients are shown too. **These are associations in a curated, biased label, not causal drivers of exploitation.**

In [ ]:
key = KIND_SPEC[SELECTED]
sel_pipe = Pipeline([("prep", PREPS[key]["prep"]), ("model", FITTED[SELECTED])])
sel_cols = PREPS[key]["cols"]
rng = np.random.default_rng(SEED)
pos_i, neg_i = va_idx[y_va == 1], va_idx[y_va == 0]
sample = np.concatenate([pos_i, rng.choice(neg_i, size=min(len(neg_i), max(CONFIG["PERM_IMPORTANCE_ROWS"] - len(pos_i), 1000)), replace=False)])
ap_scorer = lambda est, X, y: average_precision_score(y, est.predict_proba(X)[:, 1])
pi = permutation_importance(sel_pipe, df.iloc[sample][sel_cols], Y[sample], scoring=ap_scorer, n_repeats=5, random_state=SEED, n_jobs=1)
pi_df = pd.DataFrame({"feature": sel_cols, "pr_auc_drop": pi.importances_mean, "sd": pi.importances_std}).sort_values("pr_auc_drop", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
top = pi_df.head(15).iloc[::-1]
axes[0].barh(top["feature"], top["pr_auc_drop"], xerr=top["sd"], color="#3b6ea5", capsize=2); axes[0].set(title=f"Permutation importance ({SELECTED}; validation sample incl. all {len(pos_i)} positives)", xlabel="drop in PR-AUC when shuffled")
if hasattr(FITTED[SELECTED], "coef_"):
    names = PREPS[key]["prep"].get_feature_names_out(); coef = pd.Series(FITTED[SELECTED].coef_[0], index=names)
    nt = coef[~coef.index.str.startswith("text__")]; nt = pd.concat([nt.nsmallest(8), nt.nlargest(8)]).sort_values()
    axes[1].barh(nt.index.str.replace("cat__", "", regex=False).str.replace("num__", "", regex=False), nt.values, color=np.where(nt.values > 0, "#c0504d", "#4f81bd")); axes[1].set(title="Logistic-regression coefficients (non-text features)", xlabel="log-odds contribution")
    if (coef.index.str.startswith("text__")).any():
        tt = coef[coef.index.str.startswith("text__")]; print("Top description terms raising risk:", ", ".join(tt.nlargest(12).index.str.replace("text__", "", regex=False)))
else:
    axes[1].axis("off")
savefig("E11_feature_importance")
display(pi_df.head(10).round(4))

## 18. Model limitations and intended use
**Intended use (research):** produce an *ordering* of newly published CVEs by relative likelihood of KEV listing within the horizon, to study whether a review budget (e.g. top 5 %) captures most KEV-bound CVEs.

**Output semantics:** `calibrated_probability` ≈ "among CVEs resembling the training/validation distribution, the fraction later listed in KEV within the horizon". It is **not** a probability that *your* system is exploited or compromised, **not** an exploitation guarantee, and degrades under drift (KEV policy changes, new attack waves, CNA mix).

**Prohibited interpretations:** (1) "probability of compromise/breach"; (2) treating a low score as "safe" (recall is < 100 % and many positives are unpredictable from metadata); (3) treating KEV-negative as "not exploited"; (4) overriding KEV membership — if a CVE **is** in KEV it is known-exploited regardless of any model; (5) use as an EPSS replacement or for compliance decisions; (6) causal reading of feature importances.

**Dataset/label limits:** KEV is incomplete, US-federal-centric and curation-timed; a substantial share of positives is listed before/at NVD publication (quantified in §16) and is unpredictable from metadata; positives are few (hundreds), so CIs are wide; current-state NVD fields; calibration was fit on a single validation period with few positives and may not transfer; the held-out test is a single time window — **no claim of statistical significance** is made beyond the reported bootstrap/Wilson intervals.

## 19. Artifact export and reload verification
Exports one joblib **bundle** (only scikit-learn objects and primitives, so it reloads without this notebook): fitted pipeline, validation-fitted calibrator, threshold and policy. The smoke test reloads it from disk, scores raw example rows, compares with the in-memory model, checks output ranges and verifies schema rejection behaviour.

In [ ]:
MODEL_DIR = OUTPUT_DIR / "models"; MODEL_DIR.mkdir(exist_ok=True)
FEATURE_SCHEMA = {
    "schema_name": "offensivex-ml-research/cve-exploit-likelihood-input", "schema_version": "1.0.0", "status": "EXPERIMENTAL research artifact -- not for production use",
    "target": {"definition": f"CVE added to CISA KEV within {CONFIG['LABEL_HORIZON_DAYS']} days after NVD publication", "negative_means": "NOT recorded in KEV within the window (not 'never exploited')"},
    "raw_inputs": {
        "description_en": {"dtype": "string", "required": True, "missing_behavior": "REJECT",
                           "preprocessing": "sentences stating exploitation evidence are REMOVED before featurisation (regex below); inference MUST do the same",
                           "exploit_statement_pattern": EXPLOIT_STMT_PATTERN},
        "cvss_v3_vector": {"dtype": "string", "required": False, "pattern": r"^CVSS:3\.[01]/.+", "missing_behavior": "has_cvss3=0 and metric categories 'MISSING'"},
        "cvss_v3_base_score": {"dtype": "float", "required": False, "range": [0, 10], "missing_behavior": "computed from cvss_v3_vector with the official calculator; NaN if no vector"},
        "source_identifier": {"dtype": "string", "required": False, "missing_behavior": "unknown/infrequent category"},
        "primary_cwe_id": {"dtype": "string", "required": False, "missing_behavior": "unknown/infrequent category"}},
    "derived_features": {"cvss3_metrics": CVSS3_METRICS, "text_flags": FLAG_PATTERNS, "numeric": ["cvss3_score", "has_cvss3", "desc_length", "desc_words"] + FLAG_COLUMNS},
    "model_columns": sel_cols,
    "forbidden_inputs": "epss_*, kev_*, reference_count, cpe_match_count, vuln_status, last_modified, published dates, CVSS v2/v4 fields (leakage / non-stationarity)",
    "output": {"calibrated_probability": f"P(KEV listing within {CONFIG['LABEL_HORIZON_DAYS']}d | CVE resembles training distribution); NOT probability of compromise",
               "flag": f"calibrated_probability >= {THRESHOLD:.5f} (policy {CONFIG['THRESHOLD_POLICY']}, chosen on validation)"}}


def prepare_inputs(raw, schema=FEATURE_SCHEMA):
    if "description_en" not in raw.columns:
        raise ValueError("missing required input column: description_en")
    bad = [c for c in raw.columns if FORBIDDEN_RE.search(c)]
    if bad:
        raise ValueError(f"forbidden (leakage-prone) input column(s): {bad}")
    x = raw.copy()
    for c in ("cvss_v3_vector", "cvss_v3_base_score", "source_identifier", "primary_cwe_id"):
        if c not in x.columns:
            x[c] = None
    if x["description_en"].isna().any() or x["description_en"].astype(str).str.strip().eq("").any():
        raise ValueError("description_en must be non-empty for every row")
    parsed = x["cvss_v3_vector"].map(parse_cvss_vector)
    ok = parsed.map(lambda p: p is not None and p[0] in ("3.0", "3.1") and all(m in p[1] for m in CVSS3_METRICS))
    score = pd.to_numeric(x["cvss_v3_base_score"], errors="coerce")
    if (score.dropna().lt(0) | score.dropna().gt(10)).any():
        raise ValueError("cvss_v3_base_score must be within 0-10")
    calc = pd.Series([cvss3_base_score(v) if o else np.nan for v, o in zip(x["cvss_v3_vector"], ok)], index=x.index, dtype=float)
    x["cvss3_score"] = score.where(score.notna(), calc)
    x["has_cvss3"] = x["cvss3_score"].notna().astype(int)
    for m in CVSS3_METRICS:
        x[f"c3_{m}"] = [p[1][m] if o else "MISSING" for p, o in zip(parsed, ok)]
    x["description_en"] = scrub_exploitation_statements(x["description_en"].astype(str))   # same leakage guard as training
    if (x["description_en"].str.len() == 0).any():
        raise ValueError("description_en consists only of an exploitation statement; nothing left to score after the leakage guard")
    x = add_text_features(x)
    return x[schema["model_columns"]]


def score_cves(raw, bundle):
    """Reference inference: raw rows -> raw score, calibrated probability, flag."""
    x = prepare_inputs(raw)
    s = bundle["pipeline"].predict_proba(x)[:, 1]
    p = apply_calibrator(bundle["calibration_method"], bundle["calibrator"], s)
    return pd.DataFrame({"raw_score": s, "calibrated_probability": p, "flag": p >= bundle["threshold"]})


BUNDLE = {"pipeline": sel_pipe, "calibrator": calibrator, "calibration_method": CAL_METHOD, "threshold": THRESHOLD, "threshold_policy": CONFIG["THRESHOLD_POLICY"],
          "label_horizon_days": CONFIG["LABEL_HORIZON_DAYS"], "selected_model": SELECTED, "fast_dev": CONFIG["FAST_DEV"], "sklearn_version": sklearn.__version__, "model_columns": sel_cols}
joblib.dump(BUNDLE, MODEL_DIR / "exploit_likelihood_bundle.joblib", compress=3)
write_json(OUTPUT_DIR / "feature_schema.json", FEATURE_SCHEMA)
write_json(OUTPUT_DIR / "threshold.json", {"threshold": THRESHOLD, "policy": CONFIG["THRESHOLD_POLICY"], "recall_target": CONFIG["RECALL_TARGET"], "chosen_on": "validation split (calibrated probabilities)",
                                           "alternatives": {"max_f1": THR_F1, "recall_target": THR_RECALL}, "test_operating_point": dict(sel)})
write_json(OUTPUT_DIR / "target_definition.json", {"label": FEATURE_SCHEMA["target"], "horizon_days": CONFIG["LABEL_HORIZON_DAYS"], "snapshot_date": SNAPSHOT, "censoring": "CVEs published within one horizon of the snapshot are dropped",
                                                   "kev_before_publication_counted_positive": True, "embargo_days": CONFIG["LABEL_HORIZON_DAYS"], "split_table": split_table.to_dict(orient="records")})
write_json(OUTPUT_DIR / "hyperparameters.json", {"selected": SELECTED, "best_params_by_model": BEST_PARAMS, "calibration": {"method": CAL_METHOD, "cv_brier_on_validation": cv_brier}, "seed": SEED})
METRICS = {"fast_dev": CONFIG["FAST_DEV"], "selected_model": SELECTED, "split": split_table.to_dict(orient="records"), "test": {**rank_sel, "pr_auc_ci95": ci_sel["pr_auc"], "roc_auc_ci95": ci_sel["roc_auc"]},
           "brier_test": BRIER, "operating_point_test": sel, "baselines_test": BASELINES, "leakage_ablation": ABLATION.to_dict(orient="records"), "calibration_method": CAL_METHOD}
write_json(OUTPUT_DIR / "metrics.json", METRICS)
export_common(tables={"model_comparison": model_cmp, "tuning_log": pd.DataFrame(TUNING_LOG), "threshold_table_test": THR_TABLE, "baselines_test": pd.DataFrame(BASELINES).T.reset_index(names="baseline"),
                      "leakage_ablation": ABLATION, "performance_by_quarter": quarter_df, "subgroup_metrics": subgroup_df, "leakage_table": leakage_table, "permutation_importance": pi_df,
                      "false_positives_top50": fp.head(50)[cols_show].assign(description_en=lambda d: d.description_en.str.slice(0, 160)),
                      "false_negatives": fn[cols_show].assign(description_en=lambda d: d.description_en.str.slice(0, 160))},
              extra_config={"fast_dev": CONFIG["FAST_DEV"], "snapshot_date": SNAPSHOT})

loaded = joblib.load(MODEL_DIR / "exploit_likelihood_bundle.joblib")
# Reload check on rows whose raw inputs map exactly onto the evaluation frame (v3.1 vector present), so in-memory and reloaded scores must match.
chk = te_df[te_df["cvss_v31_vector"].notna()].head(6)
demo_raw = chk.rename(columns={"cvss_v31_vector": "cvss_v3_vector"})[["description_en", "source_identifier", "primary_cwe_id", "cvss_v3_vector"]].reset_index(drop=True)
out = score_cves(demo_raw, loaded)
ref = apply_calibrator(CAL_METHOD, calibrator, sel_pipe.predict_proba(chk[sel_cols])[:, 1])
assert np.allclose(out["calibrated_probability"], ref, atol=1e-6), "reloaded bundle disagrees with in-memory model"
assert out["calibrated_probability"].between(0, 1).all() and out["flag"].dtype == bool
_ = score_cves(demo_raw.drop(columns=["cvss_v3_vector", "source_identifier", "primary_cwe_id"]).head(2), loaded)   # optional fields absent -> still scores
for bad_input, why in [(demo_raw.assign(description_en="").head(1), "empty description"), (demo_raw.assign(epss_score=0.5).head(1), "leakage column supplied"), (demo_raw.assign(cvss_v3_base_score=99.0).head(1), "score out of range")]:
    try:
        prepare_inputs(bad_input)
    except ValueError:
        continue
    raise AssertionError(f"input with {why} was NOT rejected")
display(pd.concat([demo_raw[["primary_cwe_id"]], out.round(4)], axis=1))
print("SMOKE TEST PASSED: bundle reloaded from disk; raw-row inference matches the in-memory model; optional fields may be absent; invalid inputs rejected; outputs within [0, 1].")
SMOKE_OK = True

## 20. Reproducibility checklist and conclusions
Generated from this run's numbers.

In [ ]:
display(pd.DataFrame([("random seed", SEED), ("dataset fingerprint", DATA_META["dataset_fingerprint_sha256"][:24] + "..."), ("source", DATA_META["source_kind"]), ("snapshot date", str(SNAPSHOT.date())),
                      ("label horizon (days) / embargo", f"{CONFIG['LABEL_HORIZON_DAYS']} / {CONFIG['LABEL_HORIZON_DAYS']}"), ("train / val / test positives", f"{int(y_tr.sum())} / {int(y_va.sum())} / {int(y_te.sum())}"),
                      ("python / sklearn", f"{DEPENDENCY_VERSIONS['python']} / {DEPENDENCY_VERSIONS['sklearn']}"), ("FAST_DEV (must be False to report)", CONFIG["FAST_DEV"]), ("plots saved", len(SAVED_PLOTS)),
                      ("reload smoke test", SMOKE_OK), ("output dir", str(OUTPUT_DIR))], columns=["item", "value"]))
cv_ = BASELINES["rank_by_cvss3_score"]; ep_ = BASELINES["EPSS_current (hindsight-contaminated reference)"]
leak = ABLATION["test_pr_auc"].values
lines = [
    f"* **RQ1 (ranking).** Selected `{SELECTED}`: test PR-AUC {rank_sel['pr_auc']:.3f} (95% CI {ci_sel['pr_auc'][0]:.2f}-{ci_sel['pr_auc'][1]:.2f}) vs chance {PREV_TE:.4f} ({rank_sel['pr_auc'] / PREV_TE:.0f}x) and vs CVSS-score-only {cv_['pr_auc']:.3f}; ROC-AUC {rank_sel['roc_auc']:.3f}. EPSS-current (hindsight-contaminated reference, *not* a fair comparator) reaches {ep_['pr_auc']:.3f}.",
    f"* **RQ2 (calibration).** `{CAL_METHOD}` calibration fitted on validation only. Test Brier: constant-prevalence {BRIER['constant_prevalence_baseline']:.5f}, raw {BRIER['raw']:.5f}, calibrated {BRIER['calibrated']:.5f}. Reliability curve in E3 (few positives per bin -> wide uncertainty).",
    f"* **RQ3 (operating points).** Threshold {THRESHOLD:.4f} (policy `{CONFIG['THRESHOLD_POLICY']}`, chosen on validation): test recall {sel['recall']:.2f} (95% CI {sel['recall_ci95'][0]:.2f}-{sel['recall_ci95'][1]:.2f}), precision {sel['precision']:.3f}, flagging {sel['flagged_fraction']:.1%} of CVEs. Review-budget view: top-5% captures {rank_sel['recall@5%']:.0%} of KEV-bound CVEs (lift {rank_sel['lift@5%']:.1f}x); top-10% captures {rank_sel['recall@10%']:.0%}.",
    f"* **RQ4 (leakage).** HGB with leaked fields (refs, CPE counts, current EPSS) reaches test PR-AUC {leak[1]:.3f} vs {leak[0]:.3f} with honest features" + (" -- a large inflation, which is why those fields are excluded." if leak[1] > 1.15 * leak[0] else " -- no large gain on this split, but they remain excluded on principle (post-publication information).")
    + f" Leaving exploitation sentences in the description text moves the text model from {leak[2]:.3f} to {leak[3]:.3f} PR-AUC ({int(df['had_exploit_statement'].sum()):,} descriptions were scrubbed; {df.loc[df.y == 1, 'had_exploit_statement'].mean():.0%} of positives vs {df.loc[df.y == 0, 'had_exploit_statement'].mean():.2%} of negatives contained one).",
    f"* **RQ5 (errors).** {len(fp):,} false positives ({late:.0%} of them were added to KEV later, outside the window) and {len(fn):,} false negatives ({fn_pre:.0%} already listed at/before publication; {pos_pre:.0%} of all test positives). Precision is low in absolute terms because positives are rare; the value is in *ordering*, not point predictions.",
    "* **Decision for OffensiveX:** KEV membership (if known) and the deterministic risk engine remain authoritative; this model could at most add a clearly-labelled *prioritisation hint* for CVEs not yet in KEV, after stakeholder review of the precision/recall trade-off and an out-of-time re-validation. ADR-009's criterion (offline evaluation showing a model beats the deterministic engine on labelled data) is **not** met by this notebook, which evaluates exploit-listing only, not environmental risk.",
]
display(Markdown("\n".join(lines)))
report = f"""# Experiment report — exploit likelihood (KEV within {CONFIG['LABEL_HORIZON_DAYS']} days)
*Generated {datetime.now(timezone.utc).isoformat(timespec='seconds')} by notebooks/02_exploit_likelihood_prediction.ipynb. fast_dev={CONFIG['FAST_DEV']}.*

Status: EXPERIMENTAL research artifact. **Not** a probability of compromise. See model_card.md.

## Data
{DATA_META['source_description']} — fingerprint `{DATA_META['dataset_fingerprint_sha256']}`; snapshot {SNAPSHOT.date()}; see exclusion_ledger.csv and the split table in metrics.json.

## Conclusions
{chr(10).join(lines)}
"""
(OUTPUT_DIR / "experiment_report.md").write_text(report, encoding="utf-8")
card = f"""# Model card — exploit-likelihood ranking model (research prototype)
**Status:** EXPERIMENTAL. fast_dev={CONFIG['FAST_DEV']}. Model `{SELECTED}` + `{CAL_METHOD}` calibrator; sklearn {DEPENDENCY_VERSIONS['sklearn']}, python {DEPENDENCY_VERSIONS['python']}.

**Target:** CVE added to CISA KEV within {CONFIG['LABEL_HORIZON_DAYS']} days of NVD publication. Negative = not recorded in KEV within the window (NOT "not exploited").

**Intended use:** research; ordering newly published CVEs for analyst review budgets.

**Prohibited interpretations:** probability of compromise/breach; guarantee of exploitation; "low score = safe"; "KEV-negative = unexploited"; overriding KEV membership; EPSS replacement; compliance decisions; causal reading of importances.

**Test performance (chronological holdout, {int(y_te.sum())} positives):** PR-AUC {rank_sel['pr_auc']:.3f} (95% CI {ci_sel['pr_auc'][0]:.2f}-{ci_sel['pr_auc'][1]:.2f}; chance {PREV_TE:.4f}), ROC-AUC {rank_sel['roc_auc']:.3f}; at threshold {THRESHOLD:.4f}: recall {sel['recall']:.2f}, precision {sel['precision']:.3f}.

**Known biases/limits:** KEV is incomplete/US-federal-centric; positives are few; features reflect current NVD state; calibration from a single validation period; label includes CVEs listed before publication (unpredictable); drift over time.

**Data/secrets:** no credentials or raw data in the artifact; dataset metadata holds hashes only.
"""
(OUTPUT_DIR / "model_card.md").write_text(card, encoding="utf-8")
scan_for_secrets(OUTPUT_DIR)
print(f"\nExported {sum(1 for p in OUTPUT_DIR.rglob('*') if p.is_file())} files to {OUTPUT_DIR}")
print("plots:", sorted(SAVED_PLOTS))